In [1]:
# Bloque 1: Localizar la raíz del proyecto y verificar los CSV.
# Funciona desde la raíz o desde subcarpetas del proyecto.

from pathlib import Path

directorio_actual = Path.cwd().resolve()

raiz_proyecto = next(
    (
        carpeta
        for carpeta in [directorio_actual, *directorio_actual.parents]
        if (
            carpeta
            / "data"
            / "raw"
            / "DATASET_RETO_AGRO_2026"
        ).is_dir()
        and (carpeta / "src").is_dir()
    ),
    None
)

if raiz_proyecto is None:
    raise FileNotFoundError(
        "No se encontró la raíz de AgroCebada2026 desde "
        f"{directorio_actual}. "
        "El directorio de ejecución debe estar dentro del proyecto, "
        "que debe contener src y data/raw/DATASET_RETO_AGRO_2026."
    )

carpeta_datos = (
    raiz_proyecto
    / "data"
    / "raw"
    / "DATASET_RETO_AGRO_2026"
)

archivos_csv = {
    "basico": "Conjunto_datos_BASICO_AgroCebada2026.csv",
    "pro": "Conjunto_datos_PRO_AgroCebada.csv",
    "inventario_precipitacion": (
        "inventario_CHIRPS_precipitacion_mensual_2022_2025.csv"
    ),
    "inventario_temperatura": (
        "inventario_temperatura_2022_2025.csv"
    ),
    "inventario_topografia": "inventario_topografia.csv",
    "rendimiento": "ID_area_rendimiento_70_30_Reto_AgroCebada.csv",
}

rutas_csv = {}
problemas_rutas = []

print(f"Directorio de ejecución: {directorio_actual}")
print(f"Raíz del proyecto: {raiz_proyecto}")
print(f"Carpeta de datos: {carpeta_datos}\n")

for nombre, archivo in archivos_csv.items():
    coincidencias = sorted(
        ruta
        for ruta in carpeta_datos.rglob(archivo)
        if ruta.is_file()
    )

    if len(coincidencias) == 1:
        rutas_csv[nombre] = coincidencias[0]
        print(f"[OK] {nombre}")
        print(f"     {coincidencias[0].relative_to(raiz_proyecto)}")

    elif not coincidencias:
        problemas_rutas.append(
            f"No se encontró {archivo} dentro de {carpeta_datos}."
        )
        print(f"[NO ENCONTRADO] {nombre}")

    else:
        detalle = "\n".join(str(ruta) for ruta in coincidencias)
        problemas_rutas.append(
            f"Se encontraron varias copias de {archivo}:\n{detalle}"
        )
        print(f"[VARIAS COINCIDENCIAS] {nombre}")

if problemas_rutas:
    raise RuntimeError(
        "Revisar las rutas antes de continuar:\n\n"
        + "\n\n".join(problemas_rutas)
    )

print("\nVerificación completa. No se modificaron archivos.")


Directorio de ejecución: C:\Users\navae\OneDrive\Escritorio\Tec\Semestre 7\AgroCebada\AgroCebada2026\notebooks\01_eda_erick
Raíz del proyecto: C:\Users\navae\OneDrive\Escritorio\Tec\Semestre 7\AgroCebada\AgroCebada2026
Carpeta de datos: C:\Users\navae\OneDrive\Escritorio\Tec\Semestre 7\AgroCebada\AgroCebada2026\data\raw\DATASET_RETO_AGRO_2026

[OK] basico
     data\raw\DATASET_RETO_AGRO_2026\Conjunto_datos_BASICO_AgroCebada2026.csv
[OK] pro
     data\raw\DATASET_RETO_AGRO_2026\Conjunto_datos_PRO_AgroCebada.csv
[OK] inventario_precipitacion
     data\raw\DATASET_RETO_AGRO_2026\Reto_AgroCebada_CHIRPS_Precipitacion_2022_2025\inventario_CHIRPS_precipitacion_mensual_2022_2025.csv
[OK] inventario_temperatura
     data\raw\DATASET_RETO_AGRO_2026\Reto_AgroCebada_Temperatura_2022_2025_RECORTADA_CHIRPS\Reto_AgroCebada_Temperatura_2022_2025\inventario_temperatura_2022_2025.csv
[OK] inventario_topografia
     data\raw\DATASET_RETO_AGRO_2026\Reto_AgroCebada_Topografia_INEGI_CEM4\inventario_topograf

In [2]:
# Bloque 2: Cargar el CSV de rendimiento y revisar su estructura.
# No modifica los datos ni elimina columnas.

import pandas as pd
from IPython.display import display

ruta_rendimiento = rutas_csv["rendimiento"]

assert ruta_rendimiento.is_file(), (
    f"No existe el archivo: {ruta_rendimiento}. "
    "Ejecuta primero el bloque 1."
)

df_rendimiento_original = pd.read_csv(ruta_rendimiento)

print(f"Archivo leído: {ruta_rendimiento}")
print(f"Filas: {df_rendimiento_original.shape[0]}")
print(f"Columnas: {df_rendimiento_original.shape[1]}")

print("\nNombres exactos de las columnas:")
for columna in df_rendimiento_original.columns:
    print(repr(columna))

print("\nPrimeras cinco filas:")
display(df_rendimiento_original.head())

resumen_rendimiento = pd.DataFrame({
    "tipo": df_rendimiento_original.dtypes.astype(str),
    "valores_presentes": df_rendimiento_original.notna().sum(),
    "valores_faltantes": df_rendimiento_original.isna().sum(),
})

print("\nResumen por columna:")
display(resumen_rendimiento)

Archivo leído: C:\Users\navae\OneDrive\Escritorio\Tec\Semestre 7\AgroCebada\AgroCebada2026\data\raw\DATASET_RETO_AGRO_2026\ID_area_rendimiento_70_30_Reto_AgroCebada.csv
Filas: 197
Columnas: 5

Nombres exactos de las columnas:
'ID_POLIGONO'
'AREA_HA'
'RENDIMIENTO_T_HA'
'CONJUNTO'
'Unnamed: 4'

Primeras cinco filas:


,ID_POLIGONO,AREA_HA,RENDIMIENTO_T_HA,CONJUNTO,Unnamed: 4
0,AGC_001,7.61,3.50,ENTRENAMIENTO,NaN
1,AGC_002,2.31,4.63,ENTRENAMIENTO,NaN
2,AGC_003,18.68,NaN,PREDICCION,NaN
3,AGC_004,8.00,3.00,ENTRENAMIENTO,NaN
4,AGC_005,2.27,5.16,ENTRENAMIENTO,NaN



Resumen por columna:


,tipo,valores_presentes,valores_faltantes
ID_POLIGONO,object,197,0
AREA_HA,float64,197,0
RENDIMIENTO_T_HA,float64,138,59
CONJUNTO,object,197,0
Unnamed: 4,float64,0,197


In [3]:
# Bloque 3: Eliminar la columna extra vacía y verificar que los demás datos se conserven.

columna_extra = "Unnamed: 4"

assert columna_extra in df_rendimiento_original.columns, (
    f"No se encontró la columna {columna_extra}."
)
assert df_rendimiento_original[columna_extra].isna().all(), (
    "La columna contiene datos; se detuvo su eliminación."
)

df_rendimiento = df_rendimiento_original.drop(
    columns=[columna_extra]
).copy()

pd.testing.assert_frame_equal(
    df_rendimiento,
    df_rendimiento_original.loc[:, df_rendimiento.columns]
)

print(f"Filas: {df_rendimiento.shape[0]}")
print(f"Columnas: {df_rendimiento.shape[1]}")
print("\nColumnas conservadas:")
print(df_rendimiento.columns.tolist())

print("\nValores faltantes por columna:")
display(df_rendimiento.isna().sum().rename("valores_faltantes").to_frame())

print("\nVerificación correcta: solo se eliminó la columna extra.")

Filas: 197
Columnas: 4

Columnas conservadas:
['ID_POLIGONO', 'AREA_HA', 'RENDIMIENTO_T_HA', 'CONJUNTO']

Valores faltantes por columna:


,valores_faltantes
ID_POLIGONO,0
AREA_HA,0
RENDIMIENTO_T_HA,59
CONJUNTO,0



Verificación correcta: solo se eliminó la columna extra.


In [4]:
# Bloque 4: Cargar BÁSICO y PRO y revisar su estructura sin modificar los datos.

df_basico_original = pd.read_csv(rutas_csv["basico"], low_memory=False)
df_pro_original = pd.read_csv(rutas_csv["pro"], low_memory=False)

for nombre, tabla in [
    ("BÁSICO", df_basico_original),
    ("PRO", df_pro_original),
]:
    print(f"\n{'=' * 60}")
    print(f"Dataset: {nombre}")
    print(f"Filas: {tabla.shape[0]:,}")
    print(f"Columnas: {tabla.shape[1]}")

    print("\nColumnas y tipos de datos:")
    display(
        tabla.dtypes.astype(str)
        .rename("tipo")
        .to_frame()
        .rename_axis("columna")
    )

    print("\nPrimeras tres filas:")
    display(tabla.head(3))


Dataset: BÁSICO
Filas: 107,666
Columnas: 96

Columnas y tipos de datos:


,tipo
columna,
ID_POLIGONO,object
fecha_captura,object
sensor,object
ndvi_promedio,float64
ndvi_std,float64
...,...
enddi_promedio,float64
enddi_std,float64
enddi_max,float64



Primeras tres filas:


,ID_POLIGONO,fecha_captura,sensor,ndvi_promedio,ndvi_std,ndvi_max,ndvi_min,evi_promedio,evi_std,evi_max,...,dswi5_min,nddi_promedio,nddi_std,nddi_max,nddi_min,enddi_promedio,enddi_std,enddi_max,enddi_min,porcentaje_nubosidad
0,AGC_011,20/04/2022,Landsat,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,AGC_011,16/03/2024,Landsat,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,AGC_011,15/03/2024,Landsat,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0



Dataset: PRO
Filas: 47,804
Columnas: 20

Columnas y tipos de datos:


,tipo
columna,
ID_POLIGONO,object
fecha_captura,object
sensor,object
ndvi_promedio,float64
ndvi_std,float64
ndvi_max,float64
ndvi_min,float64
evi_promedio,float64
evi_std,float64



Primeras tres filas:


,ID_POLIGONO,fecha_captura,sensor,ndvi_promedio,ndvi_std,ndvi_max,ndvi_min,evi_promedio,evi_std,evi_max,evi_min,lai_promedio,lai_std,lai_max,lai_min,msavi_promedio,msavi_std,msavi_max,msavi_min,porcentaje_nubosidad
0,AGC_056,05/02/2025,Planet,0.300,0.042,0.553,0.175,0.138,0.022,0.251,0.068,1.049,0.182,2.144,0.586,0.135,0.022,0.244,0.066,0.0
1,AGC_056,08/02/2025,Planet,0.204,0.043,0.529,0.125,0.110,0.018,0.209,0.074,0.697,0.167,1.888,0.247,0.110,0.018,0.207,0.072,0.0
2,AGC_056,07/02/2025,Planet,0.241,0.043,0.534,0.154,0.134,0.018,0.224,0.098,0.826,0.159,1.858,0.404,0.135,0.018,0.230,0.098,0.0


In [5]:
# Bloque 5: Contar faltantes y nubosidad 100 por dataset y sensor sin modificar datos.

tablas_satelitales = {
    "BASICO": df_basico_original,
    "PRO": df_pro_original,
}

resumen_general_filas = []
resumen_sensor_filas = []
faltantes_columnas = []

for nombre, tabla in tablas_satelitales.items():
    columnas_indices = [
        columna for columna in tabla.columns
        if columna not in [
            "ID_POLIGONO", "fecha_captura", "sensor",
            "porcentaje_nubosidad"
        ]
    ]

    n_filas = len(tabla)
    nube = tabla["porcentaje_nubosidad"]

    resumen_general_filas.append({
        "dataset": nombre,
        "filas": n_filas,
        "parcelas": tabla["ID_POLIGONO"].nunique(),
        "celdas_NA_totales": int(tabla.isna().sum().sum()),
        "filas_con_algun_NA": int(tabla.isna().any(axis=1).sum()),
        "NA_nubosidad": int(nube.isna().sum()),
        "nubosidad_100": int(nube.eq(100).sum()),
        "porcentaje_filas_nubosidad_100": (
            nube.eq(100).sum() / n_filas * 100 if n_filas else float("nan")
        ),
    })

    detalle = pd.DataFrame({
        "dataset": nombre,
        "columna": tabla.columns,
        "tipo": tabla.dtypes.astype(str).to_numpy(),
        "cantidad_NA": tabla.isna().sum().to_numpy(),
        "porcentaje_NA": (
            tabla.isna().mean().mul(100).to_numpy()
        ),
    })
    faltantes_columnas.append(detalle)

    for sensor, grupo in tabla.groupby("sensor", dropna=False):
        nube_sensor = grupo["porcentaje_nubosidad"]
        indices = grupo[columnas_indices]

        resumen_sensor_filas.append({
            "dataset": nombre,
            "sensor": sensor,
            "filas": len(grupo),
            "NA_nubosidad": int(nube_sensor.isna().sum()),
            "nubosidad_100": int(nube_sensor.eq(100).sum()),
            "filas_sin_ningun_indice": int(
                indices.isna().all(axis=1).sum()
            ),
            "nubosidad_100_con_algun_indice": int(
                (nube_sensor.eq(100) & indices.notna().any(axis=1)).sum()
            ),
            "nubosidad_100_sin_ningun_indice": int(
                (nube_sensor.eq(100) & indices.isna().all(axis=1)).sum()
            ),
        })

resumen_general = pd.DataFrame(resumen_general_filas)
resumen_por_sensor = pd.DataFrame(resumen_sensor_filas)
resumen_NA_columnas = pd.concat(faltantes_columnas, ignore_index=True)

print("Resumen general:")
display(resumen_general.round(2))

print("\nResumen por sensor:")
display(resumen_por_sensor)


Resumen general:


,dataset,filas,parcelas,celdas_NA_totales,filas_con_algun_NA,NA_nubosidad,nubosidad_100,porcentaje_filas_nubosidad_100
0,BASICO,107666,197,5834976,107666,3056,28684,26.64
1,PRO,47804,197,192912,12057,0,12057,25.22



Resumen por sensor:


,dataset,sensor,filas,NA_nubosidad,nubosidad_100,filas_sin_ningun_indice,nubosidad_100_con_algun_indice,nubosidad_100_sin_ningun_indice
0,BASICO,Landsat,48626,3056,14521,17578,0,14521
1,BASICO,Sentinel-2,59040,0,14163,14163,0,14163
2,PRO,Planet,47804,0,12057,12057,0,12057


In [6]:
# Bloque 6: Comprobar la disponibilidad de VI6T frente a otros índices por sensor.

columnas_vi6t = [
    "vi6t_promedio", "vi6t_std", "vi6t_max", "vi6t_min"
]

columnas_otros_indices = [
    columna for columna in df_basico_original.columns
    if columna.endswith(("_promedio", "_std", "_max", "_min"))
    and columna not in columnas_vi6t
]

tiene_vi6t = df_basico_original[columnas_vi6t].notna().any(axis=1)
tiene_otros = (
    df_basico_original[columnas_otros_indices].notna().any(axis=1)
)

patron = pd.Series(
    "Sin datos en ambos grupos",
    index=df_basico_original.index,
    name="patron_disponibilidad"
)

patron.loc[tiene_vi6t & ~tiene_otros] = "Solo VI6T"
patron.loc[~tiene_vi6t & tiene_otros] = "Solo otros índices"
patron.loc[tiene_vi6t & tiene_otros] = "VI6T y otros índices"

resumen_patrones = (
    patron.value_counts()
    .rename_axis("patron")
    .reset_index(name="filas")
)

resumen_patrones["porcentaje"] = (
    resumen_patrones["filas"] / len(df_basico_original) * 100
).round(2)

patrones_por_sensor = pd.crosstab(
    df_basico_original["sensor"],
    patron,
    dropna=False
)

print("Disponibilidad de grupos de índices en BÁSICO:")
display(resumen_patrones)

print("\nPatrones por sensor:")
display(patrones_por_sensor)

print("\nFilas con los cuatro estadísticos de VI6T completos:")
print(int(df_basico_original[columnas_vi6t].notna().all(axis=1).sum()))

print("\nFilas con VI6T parcialmente completo (entre 1 y 3 valores):")
print(int(
    (
        tiene_vi6t
        & df_basico_original[columnas_vi6t].isna().any(axis=1)
    ).sum()
))


Disponibilidad de grupos de índices en BÁSICO:


,patron,filas,porcentaje
0,Solo otros índices,44877,41.68
1,Sin datos en ambos grupos,31741,29.48
2,Solo VI6T,31048,28.84



Patrones por sensor:


patron_disponibilidad,Sin datos en ambos grupos,Solo VI6T,Solo otros índices
sensor,,,
Landsat,17578,31048,0
Sentinel-2,14163,0,44877



Filas con los cuatro estadísticos de VI6T completos:
31048

Filas con VI6T parcialmente completo (entre 1 y 3 valores):
0


In [7]:
# Bloque 7: Contar NAs restantes excluyendo estructura por sensor y nubosidad 100.

resumen_restantes = []
detalle_restantes = []
mascaras_NA_restantes = {}

for nombre, tabla in {
    "BASICO": df_basico_original,
    "PRO": df_pro_original,
}.items():

    columnas_indices = [
        c for c in tabla.columns
        if c.endswith(("_promedio", "_std", "_max", "_min"))
    ]

    columnas_vi6t = [
        c for c in columnas_indices if c.startswith("vi6t_")
    ]
    columnas_otros = [
        c for c in columnas_indices if c not in columnas_vi6t
    ]

    na_originales = tabla.isna()
    na_restantes = na_originales.copy()

    # Excluir únicamente los índices que no corresponden al sensor.
    if nombre == "BASICO":
        na_restantes.loc[
            tabla["sensor"].eq("Landsat"), columnas_otros
        ] = False

        na_restantes.loc[
            tabla["sensor"].eq("Sentinel-2"), columnas_vi6t
        ] = False

    # Excluir índices faltantes cuando la nubosidad es exactamente 100.
    na_restantes.loc[
        tabla["porcentaje_nubosidad"].eq(100), columnas_indices
    ] = False

    mascaras_NA_restantes[nombre] = na_restantes

    total_original = int(na_originales.sum().sum())
    total_restante = int(na_restantes.sum().sum())

    resumen_restantes.append({
        "dataset": nombre,
        "NA_originales": total_original,
        "NA_excluidos_por_patrones": total_original - total_restante,
        "NA_restantes": total_restante,
        "filas_con_NA_restantes": int(
            na_restantes.any(axis=1).sum()
        ),
    })

    conteo_columnas = na_restantes.sum()
    for columna, cantidad in conteo_columnas.items():
        if cantidad > 0:
            detalle_restantes.append({
                "dataset": nombre,
                "columna": columna,
                "NA_restantes": int(cantidad),
            })

resumen_NA_restantes = pd.DataFrame(resumen_restantes)

detalle_NA_restantes = pd.DataFrame(
    detalle_restantes,
    columns=["dataset", "columna", "NA_restantes"]
)

print("Resumen de NAs fuera de los dos patrones:")
display(resumen_NA_restantes)

print("\nColumnas con NAs restantes:")
display(detalle_NA_restantes)

print(
    "\nTOTAL DE CELDAS NA RESTANTES:",
    f"{resumen_NA_restantes['NA_restantes'].sum():,}"
)


Resumen de NAs fuera de los dos patrones:


,dataset,NA_originales,NA_excluidos_por_patrones,NA_restantes,filas_con_NA_restantes
0,BASICO,5834976,5819676,15300,3061
1,PRO,192912,192912,0,0



Columnas con NAs restantes:


,dataset,columna,NA_restantes
0,BASICO,fapar_promedio,4
1,BASICO,fapar_std,4
2,BASICO,fapar_max,4
3,BASICO,fapar_min,4
4,BASICO,vi6t_promedio,3057
5,BASICO,vi6t_std,3057
6,BASICO,vi6t_max,3057
7,BASICO,vi6t_min,3057
8,BASICO,porcentaje_nubosidad,3056



TOTAL DE CELDAS NA RESTANTES: 15,300


In [8]:
# Bloque 8: Identificar qué NAs restantes coinciden y localizar los casos de FAPAR.

mascara_restante = mascaras_NA_restantes["BASICO"]
filas_pendientes = mascara_restante.any(axis=1)

columnas_pendientes = mascara_restante.columns[
    mascara_restante.any(axis=0)
].tolist()

diagnostico_pendientes = df_basico_original.loc[
    filas_pendientes,
    ["ID_POLIGONO", "fecha_captura", "sensor", "porcentaje_nubosidad"]
].copy()

diagnostico_pendientes["columnas_con_NA_restante"] = (
    mascara_restante.loc[filas_pendientes, columnas_pendientes]
    .apply(
        lambda fila: ", ".join(fila.index[fila].tolist()),
        axis=1
    )
)

resumen_patrones_restantes = (
    diagnostico_pendientes
    .groupby(
        ["sensor", "columnas_con_NA_restante"],
        dropna=False
    )
    .size()
    .reset_index(name="filas")
    .sort_values("filas", ascending=False)
)

columnas_fapar = [
    c for c in columnas_pendientes if c.startswith("fapar_")
]

filas_fapar = mascara_restante[columnas_fapar].any(axis=1)

casos_fapar = df_basico_original.loc[
    filas_fapar,
    [
        "ID_POLIGONO", "fecha_captura", "sensor",
        "porcentaje_nubosidad", "ndvi_promedio"
    ] + columnas_fapar
].copy()

print("Patrones de NAs restantes:")
display(resumen_patrones_restantes)

print("\nObservaciones con NAs restantes en FAPAR:")
display(casos_fapar)

Patrones de NAs restantes:


,sensor,columnas_con_NA_restante,filas
1,Landsat,"vi6t_promedio, vi6t_std, vi6t_max, vi6t_min, p...",3056
2,Sentinel-2,"fapar_promedio, fapar_std, fapar_max, fapar_min",4
0,Landsat,"vi6t_promedio, vi6t_std, vi6t_max, vi6t_min",1



Observaciones con NAs restantes en FAPAR:


,ID_POLIGONO,fecha_captura,sensor,porcentaje_nubosidad,ndvi_promedio,fapar_promedio,fapar_std,fapar_max,fapar_min
15866,AGC_086,07/12/2022,Sentinel-2,0.0,0.169,NaN,NaN,NaN,NaN
73569,AGC_019,07/12/2022,Sentinel-2,0.0,0.170,NaN,NaN,NaN,NaN
102363,AGC_173,07/12/2022,Sentinel-2,0.0,0.267,NaN,NaN,NaN,NaN
104330,AGC_103,07/12/2022,Sentinel-2,0.0,0.165,NaN,NaN,NaN,NaN


In [9]:
# Bloque 9: Localizar el caso aislado de VI6T y resumir los casos sin nubosidad.

columnas_vi6t = [
    "vi6t_promedio", "vi6t_std", "vi6t_max", "vi6t_min"
]

es_landsat = df_basico_original["sensor"].eq("Landsat")
sin_vi6t = df_basico_original[columnas_vi6t].isna().all(axis=1)
nubosidad = df_basico_original["porcentaje_nubosidad"]

columnas_revision = [
    "ID_POLIGONO", "fecha_captura", "sensor",
    "porcentaje_nubosidad"
] + columnas_vi6t

caso_vi6t_aislado = df_basico_original.loc[
    es_landsat
    & sin_vi6t
    & nubosidad.notna()
    & nubosidad.ne(100),
    columnas_revision
].copy()

casos_sin_vi6t_ni_nube = df_basico_original.loc[
    es_landsat & sin_vi6t & nubosidad.isna(),
    columnas_revision
].copy()

resumen_fechas_landsat = (
    casos_sin_vi6t_ni_nube
    .groupby("fecha_captura")
    .agg(
        filas=("ID_POLIGONO", "size"),
        parcelas=("ID_POLIGONO", "nunique")
    )
    .reset_index()
    .sort_values("filas", ascending=False)
)

print("Caso aislado: VI6T vacío con nubosidad registrada distinta de 100:")
display(caso_vi6t_aislado)

print("\nCasos sin VI6T ni dato de nubosidad:")
print(f"Filas: {len(casos_sin_vi6t_ni_nube):,}")
print(f"Parcelas: {casos_sin_vi6t_ni_nube['ID_POLIGONO'].nunique()}")
print(f"Fechas: {casos_sin_vi6t_ni_nube['fecha_captura'].nunique()}")

print("\nDiez fechas con más registros afectados:")
display(resumen_fechas_landsat.head(10))

Caso aislado: VI6T vacío con nubosidad registrada distinta de 100:


,ID_POLIGONO,fecha_captura,sensor,porcentaje_nubosidad,vi6t_promedio,vi6t_std,vi6t_max,vi6t_min
96082,AGC_013,19/05/2024,Landsat,97.3,NaN,NaN,NaN,NaN



Casos sin VI6T ni dato de nubosidad:
Filas: 3,056
Parcelas: 40
Fechas: 195

Diez fechas con más registros afectados:


,fecha_captura,filas,parcelas
128,20/09/2023,27,27
129,21/01/2023,27,27
87,14/06/2023,27,27
109,17/10/2023,27,27
2,01/05/2023,26,26
52,09/04/2023,26,26
99,16/08/2022,20,20
16,03/11/2024,20,20
36,06/10/2022,19,19
103,16/12/2025,19,19


In [10]:
# Bloque 10: Excluir filas sin nubosidad ni índices, conservando una copia de lo excluido.

columnas_indices_basico = [
    c for c in df_basico_original.columns
    if c.endswith(("_promedio", "_std", "_max", "_min"))
]

sin_nubosidad = (
    df_basico_original["porcentaje_nubosidad"].isna()
)

sin_ningun_indice = (
    df_basico_original[columnas_indices_basico]
    .isna()
    .all(axis=1)
)

assert not (sin_nubosidad & ~sin_ningun_indice).any(), (
    "Hay filas sin nubosidad que sí tienen índices. "
    "Se detuvo la limpieza para revisarlas."
)

filas_a_excluir = sin_nubosidad & sin_ningun_indice

df_basico_excluidos_sin_datos = df_basico_original.loc[
    filas_a_excluir
].copy()

df_basico = df_basico_original.loc[
    ~filas_a_excluir
].copy()

assert len(df_basico) + len(df_basico_excluidos_sin_datos) == (
    len(df_basico_original)
)

assert df_basico["porcentaje_nubosidad"].notna().all()

parcelas_sin_registros = (
    set(df_basico_original["ID_POLIGONO"].dropna())
    - set(df_basico["ID_POLIGONO"].dropna())
)

print(f"Filas originales: {len(df_basico_original):,}")
print(f"Filas excluidas: {len(df_basico_excluidos_sin_datos):,}")
print(f"Filas conservadas: {len(df_basico):,}")
print(f"Columnas conservadas: {df_basico.shape[1]}")
print(f"Parcelas conservadas: {df_basico['ID_POLIGONO'].nunique()}")

print("\nParcelas que quedaron sin ningún registro en BÁSICO:")
print(sorted(parcelas_sin_registros))

print("\nNAs restantes en nubosidad:")
print(df_basico["porcentaje_nubosidad"].isna().sum())

print("\nLimpieza realizada en memoria; los CSV originales no se modificaron.")

Filas originales: 107,666
Filas excluidas: 3,056
Filas conservadas: 104,610
Columnas conservadas: 96
Parcelas conservadas: 197

Parcelas que quedaron sin ningún registro en BÁSICO:
[]

NAs restantes en nubosidad:
0

Limpieza realizada en memoria; los CSV originales no se modificaron.


In [11]:
# Bloque 11: Evaluar valores disponibles de VI6T con nubosidad similar, sin imputar.

landsat_revision = df_basico.loc[
    df_basico["sensor"].eq("Landsat")
].copy()

nube = landsat_revision["porcentaje_nubosidad"]

rangos_nubosidad = {
    "Exactamente 97.3%": nube.eq(97.3),
    "Desde 97% hasta menos de 98%": nube.ge(97) & nube.lt(98),
    "Desde 95% hasta menos de 100%": nube.ge(95) & nube.lt(100),
}

resultados_candidatos = []

for alcance, mascara_parcela in {
    "Todas las parcelas": pd.Series(
        True, index=landsat_revision.index
    ),
    "Solo AGC_013": landsat_revision["ID_POLIGONO"].eq("AGC_013"),
}.items():
    for rango, mascara_rango in rangos_nubosidad.items():
        candidatos = landsat_revision.loc[
            mascara_parcela & mascara_rango
        ]

        for columna in columnas_vi6t:
            valores = candidatos[columna].dropna()

            resultados_candidatos.append({
                "alcance": alcance,
                "rango": rango,
                "variable": columna,
                "filas_en_rango": len(candidatos),
                "valores_disponibles": len(valores),
                "media": valores.mean(),
                "mediana": valores.median(),
                "desviacion": valores.std(),
                "minimo": valores.min(),
                "maximo": valores.max(),
            })

revision_candidatos_vi6t = pd.DataFrame(resultados_candidatos)

display(revision_candidatos_vi6t.round(4))


,alcance,rango,variable,filas_en_rango,valores_disponibles,media,mediana,desviacion,minimo,maximo
0,Todas las parcelas,Exactamente 97.3%,vi6t_promedio,2,1,-0.6670,-0.6670,NaN,-0.667,-0.667
1,Todas las parcelas,Exactamente 97.3%,vi6t_std,2,1,0.0030,0.0030,NaN,0.003,0.003
2,Todas las parcelas,Exactamente 97.3%,vi6t_max,2,1,-0.6640,-0.6640,NaN,-0.664,-0.664
3,Todas las parcelas,Exactamente 97.3%,vi6t_min,2,1,-0.6700,-0.6700,NaN,-0.670,-0.670
4,Todas las parcelas,Desde 97% hasta menos de 98%,vi6t_promedio,95,94,-0.6149,-0.6165,0.0400,-0.802,-0.517
5,Todas las parcelas,Desde 97% hasta menos de 98%,vi6t_std,95,94,0.0054,0.0035,0.0064,0.000,0.028
6,Todas las parcelas,Desde 97% hasta menos de 98%,vi6t_max,95,94,-0.6084,-0.6090,0.0422,-0.802,-0.508
7,Todas las parcelas,Desde 97% hasta menos de 98%,vi6t_min,95,94,-0.6216,-0.6280,0.0391,-0.802,-0.522
8,Todas las parcelas,Desde 95% hasta menos de 100%,vi6t_promedio,356,355,-0.6127,-0.6160,0.0393,-0.802,-0.466
9,Todas las parcelas,Desde 95% hasta menos de 100%,vi6t_std,356,355,0.0050,0.0030,0.0068,0.000,0.054


In [12]:
# Bloque 12: Imputar el caso aislado de VI6T en una copia experimental.

columnas_vi6t = [
    "vi6t_promedio", "vi6t_std", "vi6t_max", "vi6t_min"
]

fechas_basico = pd.to_datetime(
    df_basico["fecha_captura"],
    format="%d/%m/%Y",
    errors="raise"
)

es_parcela_sensor = (
    df_basico["ID_POLIGONO"].eq("AGC_013")
    & df_basico["sensor"].eq("Landsat")
)

es_objetivo = (
    es_parcela_sensor
    & fechas_basico.eq(pd.Timestamp("2024-05-19"))
    & df_basico["porcentaje_nubosidad"].eq(97.3)
)

assert es_objetivo.sum() == 1, (
    "Se esperaba exactamente una fila objetivo."
)

assert df_basico.loc[es_objetivo, columnas_vi6t].isna().all().all(), (
    "La fila objetivo ya contiene valores de VI6T."
)

es_donante = (
    es_parcela_sensor
    & df_basico["porcentaje_nubosidad"].ge(95)
    & df_basico["porcentaje_nubosidad"].lt(100)
    & ~es_objetivo
    & df_basico[columnas_vi6t].notna().all(axis=1)
)

donantes_vi6t = df_basico.loc[
    es_donante,
    ["ID_POLIGONO", "fecha_captura", "porcentaje_nubosidad"]
    + columnas_vi6t
].copy()

assert len(donantes_vi6t) >= 2, (
    "No hay al menos dos observaciones completas para calcular las medias."
)

medias_vi6t = donantes_vi6t[columnas_vi6t].mean()

df_basico_exp_vi6t = df_basico.copy()

for columna in columnas_vi6t:
    df_basico_exp_vi6t.loc[
        es_objetivo, columna
    ] = medias_vi6t[columna]

assert df_basico_exp_vi6t.loc[
    es_objetivo, columnas_vi6t
].notna().all().all()

print("Observaciones utilizadas para calcular las medias:")
display(donantes_vi6t)

print("\nFila después de la imputación experimental:")
display(
    df_basico_exp_vi6t.loc[
        es_objetivo,
        ["ID_POLIGONO", "fecha_captura", "porcentaje_nubosidad"]
        + columnas_vi6t
    ]
)

print("\nSe imputaron 4 celdas en una copia; df_basico sigue sin cambios.")

Observaciones utilizadas para calcular las medias:


,ID_POLIGONO,fecha_captura,porcentaje_nubosidad,vi6t_promedio,vi6t_std,vi6t_max,vi6t_min
95982,AGC_013,12/04/2022,97.73,-0.620,0.001,-0.618,-0.621
96013,AGC_013,06/07/2024,96.59,-0.624,0.021,-0.596,-0.647
96223,AGC_013,27/11/2024,98.86,-0.644,0.000,-0.644,-0.644



Fila después de la imputación experimental:


,ID_POLIGONO,fecha_captura,porcentaje_nubosidad,vi6t_promedio,vi6t_std,vi6t_max,vi6t_min
96082,AGC_013,19/05/2024,97.3,-0.629333,0.007333,-0.619333,-0.637333



Se imputaron 4 celdas en una copia; df_basico sigue sin cambios.


In [13]:
# Bloque 13: Buscar referencias cercanas para los cuatro casos faltantes de FAPAR.

columnas_fapar = [
    "fapar_promedio", "fapar_std", "fapar_max", "fapar_min"
]

revision_fapar = df_basico_exp_vi6t.loc[
    df_basico_exp_vi6t["sensor"].eq("Sentinel-2")
].copy()

revision_fapar["fecha_revision"] = pd.to_datetime(
    revision_fapar["fecha_captura"],
    format="%d/%m/%Y",
    errors="raise"
)

es_caso_fapar = (
    revision_fapar[columnas_fapar].isna().any(axis=1)
    & revision_fapar["porcentaje_nubosidad"].notna()
    & revision_fapar["porcentaje_nubosidad"].ne(100)
)

casos_fapar_pendientes = revision_fapar.loc[
    es_caso_fapar
].copy()

assert len(casos_fapar_pendientes) == 4, (
    "El número de casos de FAPAR cambió; revisar antes de continuar."
)

referencias_fapar = []
resumen_referencias = []

for _, caso in casos_fapar_pendientes.iterrows():
    dias = (
        revision_fapar["fecha_revision"] - caso["fecha_revision"]
    ).dt.days

    es_referencia = (
        revision_fapar["ID_POLIGONO"].eq(caso["ID_POLIGONO"])
        & revision_fapar[columnas_fapar].notna().all(axis=1)
        & revision_fapar["porcentaje_nubosidad"].eq(0)
        & dias.abs().le(30)
        & dias.ne(0)
    )

    candidatas = revision_fapar.loc[
        es_referencia,
        ["ID_POLIGONO", "fecha_captura", "porcentaje_nubosidad",
         "ndvi_promedio"] + columnas_fapar
    ].copy()

    candidatas["dias_respecto_al_faltante"] = dias.loc[es_referencia]
    candidatas["distancia_dias"] = (
        candidatas["dias_respecto_al_faltante"].abs()
    )
    candidatas["fecha_faltante"] = caso["fecha_captura"]

    resumen_referencias.append({
        "parcela": caso["ID_POLIGONO"],
        "fecha_faltante": caso["fecha_captura"],
        "referencias_anteriores": int(
            candidatas["dias_respecto_al_faltante"].lt(0).sum()
        ),
        "referencias_posteriores": int(
            candidatas["dias_respecto_al_faltante"].gt(0).sum()
        ),
    })

    referencias_fapar.append(
        candidatas.sort_values("distancia_dias").head(6)
    )

resumen_referencias_fapar = pd.DataFrame(resumen_referencias)
candidatos_fapar = pd.concat(referencias_fapar, ignore_index=True)

print("Referencias disponibles en ±30 días, con nubosidad 0:")
display(resumen_referencias_fapar)

print("\nHasta seis observaciones más cercanas por caso:")
display(candidatos_fapar)

print("\nSolo diagnóstico: no se imputó ni eliminó ningún valor.")


Referencias disponibles en ±30 días, con nubosidad 0:


,parcela,fecha_faltante,referencias_anteriores,referencias_posteriores
0,AGC_086,07/12/2022,5,5
1,AGC_019,07/12/2022,5,4
2,AGC_173,07/12/2022,4,5
3,AGC_103,07/12/2022,5,4



Hasta seis observaciones más cercanas por caso:


,ID_POLIGONO,fecha_captura,porcentaje_nubosidad,ndvi_promedio,fapar_promedio,fapar_std,fapar_max,fapar_min,dias_respecto_al_faltante,distancia_dias,fecha_faltante
0,AGC_086,02/12/2022,0.0,0.162,0.074,0.019,0.228,0.039,-5,5,07/12/2022
1,AGC_086,27/11/2022,0.0,0.161,0.085,0.019,0.251,0.055,-10,10,07/12/2022
2,AGC_086,17/12/2022,0.0,0.148,0.075,0.017,0.157,0.052,10,10,07/12/2022
3,AGC_086,22/11/2022,0.0,0.157,0.080,0.019,0.237,0.047,-15,15,07/12/2022
4,AGC_086,22/12/2022,0.0,0.122,0.068,0.016,0.143,0.039,15,15,07/12/2022
5,AGC_086,17/11/2022,0.0,0.162,0.088,0.019,0.278,0.069,-20,20,07/12/2022
6,AGC_019,02/12/2022,0.0,0.161,0.097,0.045,0.350,0.032,-5,5,07/12/2022
7,AGC_019,27/11/2022,0.0,0.162,0.103,0.043,0.349,0.049,-10,10,07/12/2022
8,AGC_019,17/12/2022,0.0,0.148,0.091,0.029,0.228,0.035,10,10,07/12/2022
9,AGC_019,22/11/2022,0.0,0.174,0.110,0.038,0.360,0.064,-15,15,07/12/2022



Solo diagnóstico: no se imputó ni eliminó ningún valor.


In [14]:
# Bloque 14: Proponer FAPAR por interpolación temporal sin modificar la base.

propuestas_fapar = []

for indice, caso in casos_fapar_pendientes.iterrows():
    dias = (
        revision_fapar["fecha_revision"] - caso["fecha_revision"]
    ).dt.days

    referencias_validas = revision_fapar.loc[
        revision_fapar["ID_POLIGONO"].eq(caso["ID_POLIGONO"])
        & revision_fapar[columnas_fapar].notna().all(axis=1)
        & revision_fapar["porcentaje_nubosidad"].eq(0)
        & dias.abs().le(30)
        & dias.ne(0)
    ].copy()

    anteriores = referencias_validas.loc[
        referencias_validas["fecha_revision"].lt(caso["fecha_revision"])
    ].sort_values("fecha_revision")

    posteriores = referencias_validas.loc[
        referencias_validas["fecha_revision"].gt(caso["fecha_revision"])
    ].sort_values("fecha_revision")

    assert not anteriores.empty and not posteriores.empty, (
        f"Faltan referencias a ambos lados para {caso['ID_POLIGONO']}."
    )

    anterior = anteriores.iloc[-1]
    posterior = posteriores.iloc[0]

    peso_posterior = (
        (caso["fecha_revision"] - anterior["fecha_revision"]).days
        / (posterior["fecha_revision"] - anterior["fecha_revision"]).days
    )

    for columna in columnas_fapar:
        if pd.isna(caso[columna]):
            estimacion = (
                anterior[columna]
                + peso_posterior * (posterior[columna] - anterior[columna])
            )

            propuestas_fapar.append({
                "indice_fila": indice,
                "parcela": caso["ID_POLIGONO"],
                "fecha_faltante": caso["fecha_captura"],
                "variable": columna,
                "fecha_anterior": anterior["fecha_captura"],
                "valor_anterior": anterior[columna],
                "fecha_posterior": posterior["fecha_captura"],
                "valor_posterior": posterior[columna],
                "valor_propuesto": estimacion,
            })

propuesta_imputacion_fapar = pd.DataFrame(propuestas_fapar)

assert len(propuesta_imputacion_fapar) == 16, (
    "Se esperaban 16 propuestas: cuatro variables en cuatro filas."
)

display(propuesta_imputacion_fapar.round(6))

print("\nSolo propuesta: no se modificó ninguna celda de la base.")


,indice_fila,parcela,fecha_faltante,variable,fecha_anterior,valor_anterior,fecha_posterior,valor_posterior,valor_propuesto
0,15866,AGC_086,07/12/2022,fapar_promedio,02/12/2022,0.074,17/12/2022,0.075,0.074333
1,15866,AGC_086,07/12/2022,fapar_std,02/12/2022,0.019,17/12/2022,0.017,0.018333
2,15866,AGC_086,07/12/2022,fapar_max,02/12/2022,0.228,17/12/2022,0.157,0.204333
3,15866,AGC_086,07/12/2022,fapar_min,02/12/2022,0.039,17/12/2022,0.052,0.043333
4,73569,AGC_019,07/12/2022,fapar_promedio,02/12/2022,0.097,17/12/2022,0.091,0.095000
5,73569,AGC_019,07/12/2022,fapar_std,02/12/2022,0.045,17/12/2022,0.029,0.039667
6,73569,AGC_019,07/12/2022,fapar_max,02/12/2022,0.350,17/12/2022,0.228,0.309333
7,73569,AGC_019,07/12/2022,fapar_min,02/12/2022,0.032,17/12/2022,0.035,0.033000
8,102363,AGC_173,07/12/2022,fapar_promedio,27/11/2022,0.213,17/12/2022,0.146,0.179500
9,102363,AGC_173,07/12/2022,fapar_std,27/11/2022,0.125,17/12/2022,0.080,0.102500



Solo propuesta: no se modificó ninguna celda de la base.


In [15]:
# Bloque 15: Aplicar las 16 imputaciones de FAPAR en una copia y registrar los cambios.

propuestas = propuesta_imputacion_fapar.copy()

assert len(propuestas) == 16, "Se esperaban exactamente 16 propuestas."
assert not propuestas.duplicated(["indice_fila", "variable"]).any(), (
    "Hay propuestas duplicadas para una misma celda."
)
assert propuestas["valor_propuesto"].notna().all(), (
    "Alguna propuesta no tiene valor."
)
assert df_basico_exp_vi6t.index.is_unique, (
    "El índice de la base no es único."
)

df_basico_exp_imputado = df_basico_exp_vi6t.copy()

for _, propuesta in propuestas.iterrows():
    indice = propuesta["indice_fila"]
    columna = propuesta["variable"]

    assert columna in columnas_fapar, "La propuesta no corresponde a FAPAR."
    assert indice in df_basico_exp_imputado.index, (
        "La fila propuesta no existe en la base."
    )

    fila = df_basico_exp_imputado.loc[indice]

    assert fila["ID_POLIGONO"] == propuesta["parcela"]
    assert fila["fecha_captura"] == propuesta["fecha_faltante"]
    assert fila["sensor"] == "Sentinel-2"
    assert pd.isna(fila[columna]), (
        "La celda ya tiene un valor; se detuvo la imputación."
    )

    df_basico_exp_imputado.at[
        indice, columna
    ] = propuesta["valor_propuesto"]

na_antes = int(df_basico_exp_vi6t.isna().sum().sum())
na_despues = int(df_basico_exp_imputado.isna().sum().sum())

assert na_antes - na_despues == 16, (
    "La reducción de NAs no coincide con las 16 imputaciones."
)

pd.testing.assert_frame_equal(
    df_basico_exp_imputado.drop(columns=columnas_fapar),
    df_basico_exp_vi6t.drop(columns=columnas_fapar)
)

registro_imputacion_fapar = propuestas.copy()
registro_imputacion_fapar["metodo"] = (
    "Interpolación lineal temporal; misma parcela y sensor; "
    "referencias con nubosidad 0 en ±30 días"
)

indices_imputados = propuestas["indice_fila"].drop_duplicates().tolist()

resultado_fapar = df_basico_exp_imputado.loc[
    indices_imputados,
    ["ID_POLIGONO", "fecha_captura", "porcentaje_nubosidad"]
    + columnas_fapar
].copy()

print("Resultado de la imputación de FAPAR:")
display(resultado_fapar)

print(f"\nCeldas imputadas: {na_antes - na_despues}")
print(f"Filas conservadas: {len(df_basico_exp_imputado):,}")
print("Los NAs estructurales y de nubosidad 100 se mantienen.")
print("Los CSV originales no se modificaron.")


Resultado de la imputación de FAPAR:


,ID_POLIGONO,fecha_captura,porcentaje_nubosidad,fapar_promedio,fapar_std,fapar_max,fapar_min
15866,AGC_086,07/12/2022,0.0,0.074333,0.018333,0.204333,0.043333
73569,AGC_019,07/12/2022,0.0,0.095000,0.039667,0.309333,0.033000
102363,AGC_173,07/12/2022,0.0,0.179500,0.102500,0.559000,0.070000
104330,AGC_103,07/12/2022,0.0,0.097667,0.038333,0.259667,0.052000



Celdas imputadas: 16
Filas conservadas: 104,610
Los NAs estructurales y de nubosidad 100 se mantienen.
Los CSV originales no se modificaron.


In [16]:
# Bloque 16: Verificar NAs restantes tras imputar, excluyendo los dos patrones conservados.

resumen_verificacion = []
detalle_verificacion = []

for nombre, tabla in {
    "BASICO_EXPERIMENTAL": df_basico_exp_imputado,
    "PRO": df_pro_original,
}.items():

    columnas_indices = [
        c for c in tabla.columns
        if c.endswith(("_promedio", "_std", "_max", "_min"))
    ]

    columnas_vi6t_revision = [
        c for c in columnas_indices if c.startswith("vi6t_")
    ]

    columnas_otros_revision = [
        c for c in columnas_indices
        if c not in columnas_vi6t_revision
    ]

    na_totales = tabla.isna()
    na_fuera_patrones = na_totales.copy()

    if nombre == "BASICO_EXPERIMENTAL":
        na_fuera_patrones.loc[
            tabla["sensor"].eq("Landsat"),
            columnas_otros_revision
        ] = False

        na_fuera_patrones.loc[
            tabla["sensor"].eq("Sentinel-2"),
            columnas_vi6t_revision
        ] = False

    na_fuera_patrones.loc[
        tabla["porcentaje_nubosidad"].eq(100),
        columnas_indices
    ] = False

    resumen_verificacion.append({
        "dataset": nombre,
        "filas": len(tabla),
        "parcelas": tabla["ID_POLIGONO"].nunique(),
        "NA_totales": int(na_totales.sum().sum()),
        "NA_fuera_de_patrones": int(na_fuera_patrones.sum().sum()),
        "filas_con_NA_fuera_de_patrones": int(
            na_fuera_patrones.any(axis=1).sum()
        ),
    })

    for columna, cantidad in na_fuera_patrones.sum().items():
        if cantidad > 0:
            detalle_verificacion.append({
                "dataset": nombre,
                "columna": columna,
                "NA_fuera_de_patrones": int(cantidad),
            })

verificacion_final_NA = pd.DataFrame(resumen_verificacion)

detalle_final_NA = pd.DataFrame(
    detalle_verificacion,
    columns=["dataset", "columna", "NA_fuera_de_patrones"]
)

display(verificacion_final_NA)

if detalle_final_NA.empty:
    print("\nVerificación correcta: no quedan NAs fuera de los dos patrones.")
else:
    print("\nSe encontraron NAs adicionales que debemos revisar:")
    display(detalle_final_NA)

print("\nNo se modificó ningún dato en este bloque.")

,dataset,filas,parcelas,NA_totales,NA_fuera_de_patrones,filas_con_NA_fuera_de_patrones
0,BASICO_EXPERIMENTAL,104610,197,5550748,0,0
1,PRO,47804,197,192912,0,0



Verificación correcta: no quedan NAs fuera de los dos patrones.

No se modificó ningún dato en este bloque.


In [17]:
# Bloque 17: Auditar fechas y claves repetidas en BÁSICO y PRO sin modificar datos.

resumen_fechas_claves = []
detalle_fechas_problematicas = {}
detalle_claves_repetidas = {}

for nombre, tabla in {
    "BASICO_EXPERIMENTAL": df_basico_exp_imputado,
    "PRO": df_pro_original,
}.items():

    fechas = pd.to_datetime(
        tabla["fecha_captura"],
        format="%d/%m/%Y",
        errors="coerce"
    )

    fecha_vacia = tabla["fecha_captura"].isna()
    fecha_no_interpretable = fechas.isna() & ~fecha_vacia

    claves = pd.DataFrame({
        "ID_POLIGONO": tabla["ID_POLIGONO"],
        "fecha_captura": fechas,
        "sensor": tabla["sensor"],
    })

    clave_completa = claves.notna().all(axis=1)
    claves_validas = claves.loc[clave_completa]

    claves_repetidas = claves_validas.duplicated(keep=False)
    indices_repetidos = claves_validas.index[claves_repetidas]

    filas_repetidas = tabla.loc[indices_repetidos].copy()

    if not filas_repetidas.empty:
        filas_repetidas["_fecha_revision"] = fechas.loc[
            indices_repetidos
        ]
        filas_repetidas = filas_repetidas.sort_values(
            ["ID_POLIGONO", "_fecha_revision", "sensor"]
        ).drop(columns="_fecha_revision")

    detalle_claves_repetidas[nombre] = filas_repetidas

    detalle_fechas_problematicas[nombre] = tabla.loc[
        fechas.isna(),
        ["ID_POLIGONO", "fecha_captura", "sensor"]
    ].copy()

    resumen_fechas_claves.append({
        "dataset": nombre,
        "filas": len(tabla),
        "fechas_vacias": int(fecha_vacia.sum()),
        "fechas_no_interpretables": int(fecha_no_interpretable.sum()),
        "fecha_minima": fechas.min(),
        "fecha_maxima": fechas.max(),
        "filas_con_clave_incompleta": int((~clave_completa).sum()),
        "filas_en_claves_repetidas": int(claves_repetidas.sum()),
        "grupos_de_clave_repetida": int(
            claves_validas.loc[claves_repetidas]
            .drop_duplicates()
            .shape[0]
        ),
        "filas_excedentes_por_clave": int(
            claves_validas.duplicated(keep="first").sum()
        ),
        "filas_identicas_excedentes": int(
            tabla.duplicated(keep="first").sum()
        ),
    })

auditoria_fechas_claves = pd.DataFrame(resumen_fechas_claves)

print("Resumen de fechas y duplicados:")
display(auditoria_fechas_claves)

print("\nSolo diagnóstico: no se modificaron las bases.")


Resumen de fechas y duplicados:


,dataset,filas,fechas_vacias,fechas_no_interpretables,fecha_minima,fecha_maxima,filas_con_clave_incompleta,filas_en_claves_repetidas,grupos_de_clave_repetida,filas_excedentes_por_clave,filas_identicas_excedentes
0,BASICO_EXPERIMENTAL,104610,0,0,2022-01-01,2025-12-31,0,0,0,0,0
1,PRO,47804,0,0,2025-01-01,2025-12-30,0,0,0,0,0



Solo diagnóstico: no se modificaron las bases.


In [18]:
# Bloque 18: Preparar fechas y verificar los IDs satelitales contra la tabla de parcelas.

assert df_rendimiento["ID_POLIGONO"].notna().all(), (
    "Hay identificadores vacíos en rendimiento."
)

assert df_rendimiento["ID_POLIGONO"].is_unique, (
    "Hay identificadores repetidos en rendimiento."
)

df_basico_preparado = df_basico_exp_imputado.copy()
df_pro_preparado = df_pro_original.copy()

ids_referencia = set(df_rendimiento["ID_POLIGONO"])

resumen_ids = []
diferencias_ids = {}

for nombre, tabla in {
    "BASICO": df_basico_preparado,
    "PRO": df_pro_preparado,
}.items():

    tabla["fecha_captura"] = pd.to_datetime(
        tabla["fecha_captura"],
        format="%d/%m/%Y",
        errors="raise"
    )

    tabla.sort_values(
        ["ID_POLIGONO", "fecha_captura", "sensor"],
        inplace=True
    )

    ids_satelitales = set(tabla["ID_POLIGONO"])

    ids_no_reconocidos = sorted(ids_satelitales - ids_referencia)
    ids_sin_observaciones = sorted(ids_referencia - ids_satelitales)

    diferencias_ids[nombre] = {
        "no_reconocidos": ids_no_reconocidos,
        "sin_observaciones": ids_sin_observaciones,
    }

    resumen_ids.append({
        "dataset": nombre,
        "parcelas_en_rendimiento": len(ids_referencia),
        "parcelas_en_satelite": len(ids_satelitales),
        "IDs_no_reconocidos": len(ids_no_reconocidos),
        "parcelas_sin_observaciones": len(ids_sin_observaciones),
        "tipo_fecha": str(tabla["fecha_captura"].dtype),
    })

verificacion_ids = pd.DataFrame(resumen_ids)

display(verificacion_ids)

for nombre, diferencias in diferencias_ids.items():
    print(f"\n{nombre}")
    print("IDs no reconocidos:", diferencias["no_reconocidos"])
    print("Parcelas sin observaciones:", diferencias["sin_observaciones"])

print("\nCopias preparadas; no se cruzaron tablas ni se guardaron archivos.")

,dataset,parcelas_en_rendimiento,parcelas_en_satelite,IDs_no_reconocidos,parcelas_sin_observaciones,tipo_fecha
0,BASICO,197,197,0,0,datetime64[ns]
1,PRO,197,197,0,0,datetime64[ns]



BASICO
IDs no reconocidos: []
Parcelas sin observaciones: []

PRO
IDs no reconocidos: []
Parcelas sin observaciones: []

Copias preparadas; no se cruzaron tablas ni se guardaron archivos.


In [19]:
# Bloque 19: Cargar y revisar los inventarios de clima y topografía antes de integrarlos.

inventarios = {
    "precipitacion": pd.read_csv(rutas_csv["inventario_precipitacion"]),
    "temperatura": pd.read_csv(rutas_csv["inventario_temperatura"]),
    "topografia": pd.read_csv(rutas_csv["inventario_topografia"]),
}

estructuras_inventarios = {}

for nombre, tabla in inventarios.items():
    estructura = pd.DataFrame({
        "columna": tabla.columns,
        "tipo": tabla.dtypes.astype(str).to_numpy(),
        "cantidad_NA": tabla.isna().sum().to_numpy(),
    })

    estructuras_inventarios[nombre] = estructura

    print(f"\n{'=' * 60}")
    print(f"Inventario: {nombre}")
    print(f"Filas: {len(tabla)} | Columnas: {tabla.shape[1]}")

    print("\nEstructura:")
    display(estructura)

    print("\nPrimeras tres filas:")
    display(tabla.head(3))

print("\nSolo revisión: no se integraron ni modificaron datos.")



Inventario: precipitacion
Filas: 48 | Columnas: 14

Estructura:


,columna,tipo,cantidad_NA
0,anio,int64,0
1,mes,int64,0
2,archivo,object,0
3,dias_esperados,int64,0
4,dias_encontrados,int64,0
5,filas,int64,0
6,columnas,int64,0
7,resolucion_grados,float64,0
8,crs,object,0
9,unidad,object,0



Primeras tres filas:


,anio,mes,archivo,dias_esperados,dias_encontrados,filas,columnas,resolucion_grados,crs,unidad,nodata,min_mm_mes,max_mm_mes,media_mm_mes_region
0,2022,1,Precipitacion_mensual/2022/PREC_2022_01.tif,31,31,72,65,0.05,EPSG:4326,mm/mes,-9999.0,0.0,131.202194,21.537550
1,2022,2,Precipitacion_mensual/2022/PREC_2022_02.tif,28,28,72,65,0.05,EPSG:4326,mm/mes,-9999.0,0.0,106.729340,12.222431
2,2022,3,Precipitacion_mensual/2022/PREC_2022_03.tif,31,31,72,65,0.05,EPSG:4326,mm/mes,-9999.0,0.0,40.774372,7.704464



Inventario: temperatura
Filas: 96 | Columnas: 13

Estructura:


...


Inventario: topografia
Filas: 2 | Columnas: 9

Estructura:


,columna,tipo,cantidad_NA
0,archivo,object,0
1,variable,object,0
2,unidad,object,0
3,resolucion_m,int64,0
4,crs,object,0
5,min,float64,0
6,max,float64,0
7,media,float64,0
8,pixeles_validos,int64,0



Primeras tres filas:


,archivo,variable,unidad,resolucion_m,crs,min,max,media,pixeles_validos
0,Elevacion_INEGI_CEM4_120m.tif,Elevacion,m s.n.m.,120,EPSG:6372,8.0,5608.000000,1828.260950,4081208
1,Pendiente_INEGI_CEM4_120m_grados.tif,Pendiente,grados,120,EPSG:6372,0.0,70.637863,10.171154,4047765



Solo revisión: no se integraron ni modificaron datos.


In [20]:
# Bloque 20: Verificar archivos de los inventarios y localizar el shapefile de parcelas.

rutas_inventarios = {
    "precipitacion": rutas_csv["inventario_precipitacion"],
    "temperatura": rutas_csv["inventario_temperatura"],
    "topografia": rutas_csv["inventario_topografia"],
}

registros_archivos = []

for nombre, inventario in inventarios.items():
    carpeta_inventario = rutas_inventarios[nombre].parent

    for _, fila in inventario.iterrows():
        ruta_relativa = Path(
            str(fila["archivo"]).replace("\\", "/")
        )
        ruta_archivo = carpeta_inventario / ruta_relativa

        registros_archivos.append({
            "inventario": nombre,
            "archivo": str(fila["archivo"]),
            "ruta": str(ruta_archivo),
            "existe": ruta_archivo.is_file(),
        })

verificacion_archivos = pd.DataFrame(registros_archivos)

resumen_archivos = (
    verificacion_archivos
    .groupby("inventario")
    .agg(
        registros_revisados=("existe", "size"),
        archivos_encontrados=("existe", "sum")
    )
    .reset_index()
)

resumen_archivos["no_encontrados"] = (
    resumen_archivos["registros_revisados"]
    - resumen_archivos["archivos_encontrados"]
)

archivos_no_encontrados = verificacion_archivos.loc[
    ~verificacion_archivos["existe"]
].copy()

rutas_shapefile = sorted(
    carpeta_datos.rglob("Parcelas_Reto_AGC_CONJUNTO.shp")
)

print("Verificación de archivos incluidos en los inventarios:")
display(resumen_archivos)

print("\nShapefiles de parcelas encontrados:")
for ruta in rutas_shapefile:
    print(ruta.relative_to(raiz_proyecto))

if not rutas_shapefile:
    print("No se encontró el shapefile dentro de la carpeta de datos.")

componentes_shapefile = []

for ruta in rutas_shapefile:
    for extension in [".shp", ".shx", ".dbf", ".prj"]:
        componente = ruta.with_suffix(extension)

        componentes_shapefile.append({
            "archivo": str(componente),
            "existe": componente.is_file(),
        })

verificacion_shapefile = pd.DataFrame(
    componentes_shapefile,
    columns=["archivo", "existe"]
)

print("\nComponentes del shapefile:")
display(verificacion_shapefile)

if not archivos_no_encontrados.empty:
    print("\nArchivos de inventario no encontrados (primeros 10):")
    display(archivos_no_encontrados.head(10))

print("\nSolo verificación de rutas; no se modificaron archivos.")

Verificación de archivos incluidos en los inventarios:


,inventario,registros_revisados,archivos_encontrados,no_encontrados
0,precipitacion,48,48,0
1,temperatura,96,96,0
2,topografia,2,2,0



Shapefiles de parcelas encontrados:
data\raw\DATASET_RETO_AGRO_2026\Parcelas_Reto_AGC_CONJUNTO_70_30\Parcelas_Reto_AGC_CONJUNTO.shp

Componentes del shapefile:


,archivo,existe
0,C:\Users\navae\OneDrive\Escritorio\Tec\Semestr...,True
1,C:\Users\navae\OneDrive\Escritorio\Tec\Semestr...,True
2,C:\Users\navae\OneDrive\Escritorio\Tec\Semestr...,True
3,C:\Users\navae\OneDrive\Escritorio\Tec\Semestr...,True



Solo verificación de rutas; no se modificaron archivos.


In [21]:
#Instalar GeoPandas

#%pip install geopandas

In [22]:
# Bloque 21: Abrir el shapefile y revisar la estructura y geometrías de las parcelas.

import geopandas as gpd

assert len(rutas_shapefile) == 1, (
    "Se esperaba un solo shapefile de parcelas; revisar las rutas encontradas."
)

ruta_parcelas = rutas_shapefile[0]
gdf_parcelas_original = gpd.read_file(ruta_parcelas)

print(f"Parcelas leídas: {len(gdf_parcelas_original)}")
print(f"Sistema de coordenadas: {gdf_parcelas_original.crs}")

print("\nColumnas disponibles:")
print(gdf_parcelas_original.columns.tolist())

print("\nTipos de geometría:")
display(
    gdf_parcelas_original.geom_type
    .value_counts(dropna=False)
    .rename("cantidad")
    .to_frame()
)

resumen_geometrias = pd.DataFrame([{
    "filas": len(gdf_parcelas_original),
    "geometrias_nulas": int(
        gdf_parcelas_original.geometry.isna().sum()
    ),
    "geometrias_vacias": int(
        gdf_parcelas_original.geometry.is_empty.sum()
    ),
    "geometrias_no_validas_no_nulas": int(
        (
            gdf_parcelas_original.geometry.notna()
            & ~gdf_parcelas_original.geometry.is_valid
        ).sum()
    ),
}])

print("\nRevisión de geometrías:")
display(resumen_geometrias)

print("\nPrimeras cinco filas, sin mostrar la geometría:")
display(
    gdf_parcelas_original.drop(
        columns=gdf_parcelas_original.geometry.name
    ).head()
)

print("\nSolo lectura: no se modificaron archivos ni se extrajeron valores.")


Parcelas leídas: 197
Sistema de coordenadas: EPSG:4326

Columnas disponibles:
['Cultivo', 'ID_POLIGON', 'área_ha', 'Municipio', 'Estado', 'CONJUNTO', 'geometry']

Tipos de geometría:


,cantidad
Polygon,197



Revisión de geometrías:


,filas,geometrias_nulas,geometrias_vacias,geometrias_no_validas_no_nulas
0,197,0,0,0



Primeras cinco filas, sin mostrar la geometría:


,Cultivo,ID_POLIGON,área_ha,Municipio,Estado,CONJUNTO
0,Cebada,AGC_196,8.57,Emiliano Zapata,Hidalgo,ENTRENAMIENTO
1,Cebada,AGC_044,0.68,Emiliano Zapata,Hidalgo,ENTRENAMIENTO
2,Cebada,AGC_020,8.34,Emiliano Zapata,Hidalgo,ENTRENAMIENTO
3,CEBADA,AGC_174,6.89,Cuautepec de Hinojosa,Hidalgo,PREDICCION
4,CEBADA,AGC_083,2.87,Cuautepec de Hinojosa,Hidalgo,ENTRENAMIENTO



Solo lectura: no se modificaron archivos ni se extrajeron valores.


In [23]:
# Bloque 22: Alinear el ID del shapefile y comprobar su correspondencia con rendimiento.

gdf_parcelas = gdf_parcelas_original.rename(
    columns={"ID_POLIGON": "ID_POLIGONO"}
).copy()

assert gdf_parcelas["ID_POLIGONO"].notna().all(), (
    "Hay IDs vacíos en el shapefile."
)

assert gdf_parcelas["ID_POLIGONO"].is_unique, (
    "Hay IDs repetidos en el shapefile."
)

ids_geometrias = set(gdf_parcelas["ID_POLIGONO"])
ids_rendimiento = set(df_rendimiento["ID_POLIGONO"])

ids_solo_geometrias = sorted(ids_geometrias - ids_rendimiento)
ids_sin_geometria = sorted(ids_rendimiento - ids_geometrias)

print(f"Parcelas con geometría: {len(ids_geometrias)}")
print(f"Parcelas en rendimiento: {len(ids_rendimiento)}")
print("\nIDs del shapefile no reconocidos:", ids_solo_geometrias)
print("Parcelas de rendimiento sin geometría:", ids_sin_geometria)

assert not ids_solo_geometrias and not ids_sin_geometria, (
    "Los identificadores no coinciden entre ambas fuentes."
)

comparacion_conjuntos = (
    gdf_parcelas[["ID_POLIGONO", "CONJUNTO"]]
    .merge(
        df_rendimiento[["ID_POLIGONO", "CONJUNTO"]],
        on="ID_POLIGONO",
        how="inner",
        validate="one_to_one",
        suffixes=("_mapa", "_csv")
    )
)

diferencias_conjunto = comparacion_conjuntos.loc[
    comparacion_conjuntos["CONJUNTO_mapa"].ne(
        comparacion_conjuntos["CONJUNTO_csv"]
    )
]

print(
    "\nDiferencias en ENTRENAMIENTO/PREDICCION:",
    len(diferencias_conjunto)
)

if not diferencias_conjunto.empty:
    display(diferencias_conjunto)

print("\nSe creó gdf_parcelas en memoria; el shapefile original sigue intacto.")

Parcelas con geometría: 197
Parcelas en rendimiento: 197

IDs del shapefile no reconocidos: []
Parcelas de rendimiento sin geometría: []

Diferencias en ENTRENAMIENTO/PREDICCION: 0

Se creó gdf_parcelas en memoria; el shapefile original sigue intacto.


In [24]:
# Instalar Rasterio

#%pip install rasterio

In [25]:
# Bloque 23: Abrir un ráster por variable y revisar sus metadatos sin modificarlo.

import rasterio

muestras_raster = []

for nombre, inventario in inventarios.items():
    if nombre == "precipitacion":
        muestras = inventario.head(1)
    else:
        muestras = inventario.groupby(
            "variable", sort=False
        ).head(1)

    for _, fila in muestras.iterrows():
        variable = (
            "Precipitacion"
            if nombre == "precipitacion"
            else fila["variable"]
        )

        ruta = rutas_inventarios[nombre].parent / Path(
            str(fila["archivo"]).replace("\\", "/")
        )

        with rasterio.open(ruta) as raster:
            muestras_raster.append({
                "fuente": nombre,
                "variable": variable,
                "archivo": ruta.name,
                "CRS": str(raster.crs),
                "bandas": raster.count,
                "filas": raster.height,
                "columnas": raster.width,
                "resolucion_x": raster.res[0],
                "resolucion_y": raster.res[1],
                "nodata": raster.nodata,
                "tipo_dato": str(raster.dtypes),
                "escalas": str(raster.scales),
                "desplazamientos": str(raster.offsets),
                "limites": str(tuple(raster.bounds)),
            })

revision_raster_muestra = pd.DataFrame(muestras_raster)

display(revision_raster_muestra)

print(f"\nRásteres abiertos correctamente: {len(revision_raster_muestra)}")
print("Solo lectura de metadatos; no se extrajeron valores por parcela.")

,fuente,variable,archivo,CRS,bandas,filas,columnas,resolucion_x,resolucion_y,nodata,tipo_dato,escalas,desplazamientos,limites
0,precipitacion,Precipitacion,PREC_2022_01.tif,EPSG:4326,1,72,65,0.05,0.05,-9999.0,"('float32',)","(1.0,)","(0.0,)","(-99.94999880716205, 17.79999952018261, -96.69..."
1,temperatura,Tmax,Tmax_2022_01.tif,EPSG:4326,1,72,65,0.05,0.05,-9999.0,"('float32',)","(1.0,)","(0.0,)","(-99.94999880716205, 17.799999222159386, -96.6..."
2,temperatura,Tmin,Tmin_2022_01.tif,EPSG:4326,1,72,65,0.05,0.05,-9999.0,"('float32',)","(1.0,)","(0.0,)","(-99.94999880716205, 17.799999222159386, -96.6..."
3,topografia,Elevacion,Elevacion_INEGI_CEM4_120m.tif,EPSG:6372,1,3362,2900,120.00,120.00,-9999.0,"('float32',)","(1.0,)","(0.0,)","(2715840.0, 650400.0, 3063840.0, 1053840.0)"
4,topografia,Pendiente,Pendiente_INEGI_CEM4_120m_grados.tif,EPSG:6372,1,3362,2900,120.00,120.00,-9999.0,"('float32',)","(1.0,)","(0.0,)","(2715840.0, 650400.0, 3063840.0, 1053840.0)"



Rásteres abiertos correctamente: 5
Solo lectura de metadatos; no se extrajeron valores por parcela.


In [26]:
# Bloque 24: Comprobar la cobertura de las parcelas en los cinco rásteres de muestra.

from shapely.geometry import box

resumen_cobertura = []
detalle_cobertura = []

for nombre, inventario in inventarios.items():
    if nombre == "precipitacion":
        muestras = inventario.head(1)
    else:
        muestras = inventario.groupby(
            "variable", sort=False
        ).head(1)

    for _, fila in muestras.iterrows():
        variable = (
            "Precipitacion"
            if nombre == "precipitacion"
            else fila["variable"]
        )

        ruta = rutas_inventarios[nombre].parent / Path(
            str(fila["archivo"]).replace("\\", "/")
        )

        with rasterio.open(ruta) as raster:
            assert raster.crs is not None, (
                f"El ráster {ruta.name} no tiene CRS."
            )

            parcelas_en_crs = gdf_parcelas.to_crs(raster.crs)
            limite_raster = box(*raster.bounds)

            dentro_completo = parcelas_en_crs.geometry.apply(
                limite_raster.covers
            )

            intersecta = parcelas_en_crs.geometry.intersects(
                limite_raster
            )

            cobertura_parcial = intersecta & ~dentro_completo
            fuera = ~intersecta

            resumen_cobertura.append({
                "variable": variable,
                "archivo": ruta.name,
                "parcelas_revisadas": len(parcelas_en_crs),
                "dentro_completo": int(dentro_completo.sum()),
                "cobertura_parcial_o_contacto": int(
                    cobertura_parcial.sum()
                ),
                "fuera": int(fuera.sum()),
            })

            for indice in parcelas_en_crs.index[~dentro_completo]:
                detalle_cobertura.append({
                    "variable": variable,
                    "parcela": parcelas_en_crs.at[
                        indice, "ID_POLIGONO"
                    ],
                    "situacion": (
                        "Parcial o contacto con el límite"
                        if intersecta.loc[indice]
                        else "Fuera"
                    ),
                })

verificacion_cobertura = pd.DataFrame(resumen_cobertura)

problemas_cobertura = pd.DataFrame(
    detalle_cobertura,
    columns=["variable", "parcela", "situacion"]
)

display(verificacion_cobertura)

if problemas_cobertura.empty:
    print("\nTodas las parcelas están completamente dentro de los mapas de muestra.")
else:
    print("\nParcelas que requieren revisión:")
    display(problemas_cobertura)

print("\nSolo cobertura geométrica; todavía no se revisaron los píxeles nodata.")

,variable,archivo,parcelas_revisadas,dentro_completo,cobertura_parcial_o_contacto,fuera
0,Precipitacion,PREC_2022_01.tif,197,197,0,0
1,Tmax,Tmax_2022_01.tif,197,197,0,0
2,Tmin,Tmin_2022_01.tif,197,197,0,0
3,Elevacion,Elevacion_INEGI_CEM4_120m.tif,197,197,0,0
4,Pendiente,Pendiente_INEGI_CEM4_120m_grados.tif,197,197,0,0



Todas las parcelas están completamente dentro de los mapas de muestra.

Solo cobertura geométrica; todavía no se revisaron los píxeles nodata.


In [27]:
# Bloque 25: Revisar píxeles válidos por parcela comparando dos criterios de selección.

import numpy as np
from rasterio.mask import mask

registros_pixeles = []

for nombre, inventario in inventarios.items():
    if nombre == "precipitacion":
        muestras = inventario.head(1)
    else:
        muestras = inventario.groupby(
            "variable", sort=False
        ).head(1)

    for _, fila in muestras.iterrows():
        variable = (
            "Precipitacion"
            if nombre == "precipitacion"
            else fila["variable"]
        )

        ruta = rutas_inventarios[nombre].parent / Path(
            str(fila["archivo"]).replace("\\", "/")
        )

        with rasterio.open(ruta) as raster:
            parcelas_en_crs = gdf_parcelas.to_crs(raster.crs)

            for _, parcela in parcelas_en_crs.iterrows():
                registro = {
                    "variable": variable,
                    "archivo": ruta.name,
                    "ID_POLIGONO": parcela["ID_POLIGONO"],
                }

                for criterio, tocar_todos in [
                    ("predeterminado", False),
                    ("todos_los_tocados", True),
                ]:
                    recorte, _ = mask(
                        raster,
                        [parcela.geometry.__geo_interface__],
                        indexes=1,
                        crop=True,
                        filled=False,
                        all_touched=tocar_todos
                    )

                    valores = recorte.compressed()
                    valores_validos = valores[np.isfinite(valores)]

                    registro[f"pixeles_validos_{criterio}"] = (
                        len(valores_validos)
                    )

                registros_pixeles.append(registro)

diagnostico_pixeles = pd.DataFrame(registros_pixeles)

resumen_pixeles = (
    diagnostico_pixeles
    .groupby("variable")
    .agg(
        parcelas_revisadas=("ID_POLIGONO", "size"),
        sin_validos_predeterminado=(
            "pixeles_validos_predeterminado",
            lambda serie: int(serie.eq(0).sum())
        ),
        sin_validos_todos_los_tocados=(
            "pixeles_validos_todos_los_tocados",
            lambda serie: int(serie.eq(0).sum())
        ),
    )
    .reset_index()
)

display(resumen_pixeles)

print("\nNo se calcularon promedios ni se modificaron las bases.")
print("La revisión corresponde únicamente a los cinco rásteres de muestra.")

,variable,parcelas_revisadas,sin_validos_predeterminado,sin_validos_todos_los_tocados
0,Elevacion,197,17,0
1,Pendiente,197,18,0
2,Precipitacion,197,196,0
3,Tmax,197,196,0
4,Tmin,197,196,0



No se calcularon promedios ni se modificaron las bases.
La revisión corresponde únicamente a los cinco rásteres de muestra.


In [28]:
# Instalar exactextract
# %pip install exactextract

In [29]:
# Bloque 26: Extraer valores de prueba por parcela con ponderación por cobertura del píxel.

from exactextract import exact_extract

resultados_extraccion_prueba = []

for nombre, inventario in inventarios.items():
    if nombre == "precipitacion":
        muestras = inventario.head(1)
    else:
        muestras = inventario.groupby(
            "variable", sort=False
        ).head(1)

    for _, fila in muestras.iterrows():
        variable = (
            "Precipitacion"
            if nombre == "precipitacion"
            else fila["variable"]
        )

        ruta = rutas_inventarios[nombre].parent / Path(
            str(fila["archivo"]).replace("\\", "/")
        )

        with rasterio.open(ruta) as raster:
            parcelas_en_crs = gdf_parcelas[
                ["ID_POLIGONO", "geometry"]
            ].to_crs(raster.crs)

            resultado = exact_extract(
                raster,
                parcelas_en_crs,
                ["mean", "min", "max"],
                include_cols=["ID_POLIGONO"],
                output="pandas"
            )

        resultado = resultado.rename(columns={
            "mean": "media_ponderada",
            "min": "minimo_pixeles",
            "max": "maximo_pixeles",
        })

        resultado["variable"] = variable
        resultado["archivo"] = ruta.name
        resultado["anio"] = fila.get("anio", pd.NA)
        resultado["mes"] = fila.get("mes", pd.NA)

        resultados_extraccion_prueba.append(resultado)

extraccion_prueba = pd.concat(
    resultados_extraccion_prueba,
    ignore_index=True
)

assert not extraccion_prueba.duplicated(
    ["variable", "ID_POLIGONO"]
).any(), "Se encontraron resultados repetidos por parcela y variable."

extraccion_prueba["media_ponderada"] = (
    extraccion_prueba["media_ponderada"]
    .replace([np.inf, -np.inf], np.nan)
)

resumen_extraccion_prueba = (
    extraccion_prueba
    .groupby("variable")
    .agg(
        parcelas=("ID_POLIGONO", "nunique"),
        valores_disponibles=("media_ponderada", "count"),
        valores_faltantes=(
            "media_ponderada",
            lambda serie: int(serie.isna().sum())
        ),
        menor_media=("media_ponderada", "min"),
        mayor_media=("media_ponderada", "max"),
    )
    .reset_index()
)

print("Resumen de la extracción de prueba:")
display(resumen_extraccion_prueba)

print("\nPrimeros resultados:")
display(extraccion_prueba.head())

print("\nSe creó una tabla de prueba en memoria.")
print("No se modificaron las bases satelitales ni los archivos originales.")

Resumen de la extracción de prueba:


,variable,parcelas,valores_disponibles,valores_faltantes,menor_media,mayor_media
0,Elevacion,197,197,0,2496.648468,2906.039520
1,Pendiente,197,197,0,0.281865,9.396395
2,Precipitacion,197,197,0,6.847781,23.235132
3,Tmax,197,197,0,19.918749,20.436686
4,Tmin,197,197,0,6.046129,7.702507



Primeros resultados:


,ID_POLIGONO,media_ponderada,minimo_pixeles,maximo_pixeles,variable,archivo,anio,mes
0,AGC_196,6.847781,6.847781,6.847781,Precipitacion,PREC_2022_01.tif,2022,1
1,AGC_044,8.457767,8.457767,8.457767,Precipitacion,PREC_2022_01.tif,2022,1
2,AGC_020,8.457767,8.457767,8.457767,Precipitacion,PREC_2022_01.tif,2022,1
3,AGC_174,19.384503,19.384502,23.592558,Precipitacion,PREC_2022_01.tif,2022,1
4,AGC_083,21.224924,21.224924,21.224924,Precipitacion,PREC_2022_01.tif,2022,1



Se creó una tabla de prueba en memoria.
No se modificaron las bases satelitales ni los archivos originales.


In [30]:
# Bloque 27: Extraer el clima mensual de todos los archivos para las 197 parcelas.

resultados_clima = []
parcelas_por_crs = {}
archivos_procesados = 0

total_archivos_clima = (
    len(inventarios["precipitacion"])
    + len(inventarios["temperatura"])
)

for nombre in ["precipitacion", "temperatura"]:
    inventario = inventarios[nombre]

    clave_inventario = (
        ["anio", "mes"]
        if nombre == "precipitacion"
        else ["variable", "anio", "mes"]
    )

    assert not inventario.duplicated(clave_inventario).any(), (
        f"Hay periodos repetidos en el inventario de {nombre}."
    )

    for _, fila in inventario.iterrows():
        variable = (
            "Precipitacion"
            if nombre == "precipitacion"
            else fila["variable"]
        )

        ruta = rutas_inventarios[nombre].parent / Path(
            str(fila["archivo"]).replace("\\", "/")
        )

        with rasterio.open(ruta) as raster:
            assert raster.crs is not None, f"Falta CRS en {ruta.name}."
            assert raster.count == 1, (
                f"Se esperaba una sola banda en {ruta.name}."
            )
            assert raster.scales[0] == 1 and raster.offsets[0] == 0, (
                f"Hay una escala o desplazamiento por revisar en {ruta.name}."
            )

            clave_crs = raster.crs.to_wkt()

            if clave_crs not in parcelas_por_crs:
                parcelas_por_crs[clave_crs] = (
                    gdf_parcelas[["ID_POLIGONO", "geometry"]]
                    .to_crs(raster.crs)
                )

            resultado = exact_extract(
                raster,
                parcelas_por_crs[clave_crs],
                ["mean", "min", "max"],
                include_cols=["ID_POLIGONO"],
                output="pandas"
            )

        assert len(resultado) == len(gdf_parcelas), (
            f"El número de parcelas no coincide en {ruta.name}."
        )

        resultado = resultado.rename(columns={
            "mean": "media_ponderada",
            "min": "minimo_pixeles",
            "max": "maximo_pixeles",
        })

        resultado["variable"] = variable
        resultado["anio"] = int(fila["anio"])
        resultado["mes"] = int(fila["mes"])
        resultado["archivo"] = ruta.name

        resultados_clima.append(resultado)
        archivos_procesados += 1

        if archivos_procesados % 12 == 0:
            print(
                f"Archivos procesados: "
                f"{archivos_procesados}/{total_archivos_clima}"
            )

df_clima_mensual_largo = pd.concat(
    resultados_clima, ignore_index=True
)

columnas_valores = [
    "media_ponderada", "minimo_pixeles", "maximo_pixeles"
]

df_clima_mensual_largo[columnas_valores] = (
    df_clima_mensual_largo[columnas_valores]
    .replace([np.inf, -np.inf], np.nan)
)

assert not df_clima_mensual_largo.duplicated(
    ["ID_POLIGONO", "anio", "mes", "variable"]
).any(), "Hay claves repetidas en la tabla climática."

resumen_clima_extraido = (
    df_clima_mensual_largo
    .groupby("variable")
    .agg(
        registros=("ID_POLIGONO", "size"),
        parcelas=("ID_POLIGONO", "nunique"),
        valores_faltantes=(
            "media_ponderada",
            lambda serie: int(serie.isna().sum())
        ),
        menor_media=("media_ponderada", "min"),
        mayor_media=("media_ponderada", "max"),
    )
    .reset_index()
)

print("\nResumen de la extracción climática:")
display(resumen_clima_extraido)

print(f"\nFilas generadas: {len(df_clima_mensual_largo):,}")
print("Tabla creada en memoria; aún no se integró con otras bases.")


Archivos procesados: 12/144
Archivos procesados: 24/144
Archivos procesados: 36/144
Archivos procesados: 48/144
Archivos procesados: 60/144
Archivos procesados: 72/144
Archivos procesados: 84/144
Archivos procesados: 96/144
Archivos procesados: 108/144
Archivos procesados: 120/144
Archivos procesados: 132/144
Archivos procesados: 144/144

Resumen de la extracción climática:


,variable,registros,parcelas,valores_faltantes,menor_media,mayor_media
0,Precipitacion,9456,197,0,0.000000,392.024017
1,Tmax,9456,197,0,18.760885,30.158184
2,Tmin,9456,197,0,5.415943,16.259834



Filas generadas: 28,368
Tabla creada en memoria; aún no se integró con otras bases.


In [31]:
# Bloque 28: Organizar el clima por parcela y mes y comprobar cobertura y consistencia.

clave_clima = ["ID_POLIGONO", "anio", "mes", "variable"]

assert not df_clima_mensual_largo.duplicated(clave_clima).any(), (
    "Hay claves climáticas repetidas."
)

df_clima_mensual = (
    df_clima_mensual_largo
    .pivot(
        index=["ID_POLIGONO", "anio", "mes"],
        columns="variable",
        values="media_ponderada"
    )
    .rename(columns={
        "Precipitacion": "precipitacion_mm",
        "Tmax": "tmax_c",
        "Tmin": "tmin_c",
    })
    .reset_index()
)

df_clima_mensual.columns.name = None

df_clima_mensual["fecha_mes"] = pd.to_datetime({
    "year": df_clima_mensual["anio"],
    "month": df_clima_mensual["mes"],
    "day": 1,
})

df_clima_mensual = df_clima_mensual[
    [
        "ID_POLIGONO", "anio", "mes", "fecha_mes",
        "precipitacion_mm", "tmax_c", "tmin_c"
    ]
].sort_values(["ID_POLIGONO", "fecha_mes"]).reset_index(drop=True)

combinaciones_esperadas = pd.MultiIndex.from_product(
    [
        sorted(df_rendimiento["ID_POLIGONO"]),
        pd.date_range("2022-01-01", "2025-12-01", freq="MS"),
    ],
    names=["ID_POLIGONO", "fecha_mes"]
)

combinaciones_obtenidas = pd.MultiIndex.from_frame(
    df_clima_mensual[["ID_POLIGONO", "fecha_mes"]]
)

combinaciones_faltantes = combinaciones_esperadas.difference(
    combinaciones_obtenidas
)

combinaciones_extra = combinaciones_obtenidas.difference(
    combinaciones_esperadas
)

columnas_clima = ["precipitacion_mm", "tmax_c", "tmin_c"]

resumen_clima_organizado = pd.DataFrame([{
    "filas": len(df_clima_mensual),
    "parcelas": df_clima_mensual["ID_POLIGONO"].nunique(),
    "meses": df_clima_mensual["fecha_mes"].nunique(),
    "combinaciones_faltantes": len(combinaciones_faltantes),
    "combinaciones_extra": len(combinaciones_extra),
    "NA_climaticos": int(
        df_clima_mensual[columnas_clima].isna().sum().sum()
    ),
    "precipitaciones_negativas": int(
        df_clima_mensual["precipitacion_mm"].lt(0).sum()
    ),
    "casos_Tmin_mayor_Tmax": int(
        df_clima_mensual["tmin_c"].gt(
            df_clima_mensual["tmax_c"]
        ).sum()
    ),
}])

print("Revisión de la tabla climática:")
display(resumen_clima_organizado)

print("\nPrimeras cinco filas:")
display(df_clima_mensual.head())

print("\nTabla creada en memoria; aún no se cruzó con satélite ni rendimiento.")

Revisión de la tabla climática:


,filas,parcelas,meses,combinaciones_faltantes,combinaciones_extra,NA_climaticos,precipitaciones_negativas,casos_Tmin_mayor_Tmax
0,9456,197,48,0,0,0,0,0



Primeras cinco filas:


,ID_POLIGONO,anio,mes,fecha_mes,precipitacion_mm,tmax_c,tmin_c
0,AGC_001,2022,1,2022-01-01,19.545547,20.292349,6.107370
1,AGC_001,2022,2,2022-02-01,14.697198,21.410572,6.613323
2,AGC_001,2022,3,2022-03-01,4.974521,24.541271,8.275019
3,AGC_001,2022,4,2022-04-01,47.755169,25.377922,11.046246
4,AGC_001,2022,5,2022-05-01,45.257622,26.147617,11.043602



Tabla creada en memoria; aún no se cruzó con satélite ni rendimiento.


In [32]:
# Bloque 29: Organizar topografía por parcela e integrarla con el clima mensual.

topografia_extraida = extraccion_prueba.loc[
    extraccion_prueba["variable"].isin(["Elevacion", "Pendiente"]),
    ["ID_POLIGONO", "variable", "media_ponderada"]
].copy()

assert not topografia_extraida.duplicated(
    ["ID_POLIGONO", "variable"]
).any(), "Hay registros topográficos repetidos."

df_topografia = (
    topografia_extraida
    .pivot(
        index="ID_POLIGONO",
        columns="variable",
        values="media_ponderada"
    )
    .rename(columns={
        "Elevacion": "elevacion_m",
        "Pendiente": "pendiente_grados",
    })
    .reset_index()
)

df_topografia.columns.name = None

assert set(df_topografia["ID_POLIGONO"]) == set(
    df_rendimiento["ID_POLIGONO"]
), "Los IDs topográficos no coinciden con los de rendimiento."

assert df_topografia[
    ["elevacion_m", "pendiente_grados"]
].notna().all().all(), "Hay valores topográficos faltantes."

df_ambiente_mensual = df_clima_mensual.merge(
    df_topografia,
    on="ID_POLIGONO",
    how="left",
    validate="many_to_one",
    indicator=True
)

assert df_ambiente_mensual["_merge"].eq("both").all(), (
    "Alguna fila climática no encontró topografía."
)

df_ambiente_mensual = df_ambiente_mensual.drop(columns="_merge")

assert len(df_ambiente_mensual) == len(df_clima_mensual), (
    "El cruce cambió el número de filas."
)

assert not df_ambiente_mensual.duplicated(
    ["ID_POLIGONO", "fecha_mes"]
).any(), "La tabla integrada tiene claves repetidas."

resumen_ambiente = pd.DataFrame([{
    "filas": len(df_ambiente_mensual),
    "parcelas": df_ambiente_mensual["ID_POLIGONO"].nunique(),
    "meses": df_ambiente_mensual["fecha_mes"].nunique(),
    "columnas": df_ambiente_mensual.shape[1],
    "NA_totales": int(df_ambiente_mensual.isna().sum().sum()),
}])

print("Revisión de clima y topografía integrados:")
display(resumen_ambiente)

print("\nPrimeras cinco filas:")
display(df_ambiente_mensual.head())

print("\nSe creó df_ambiente_mensual en memoria.")
print("No se modificaron las tablas de origen ni se guardaron archivos.")


Revisión de clima y topografía integrados:


,filas,parcelas,meses,columnas,NA_totales
0,9456,197,48,9,0



Primeras cinco filas:


,ID_POLIGONO,anio,mes,fecha_mes,precipitacion_mm,tmax_c,tmin_c,elevacion_m,pendiente_grados
0,AGC_001,2022,1,2022-01-01,19.545547,20.292349,6.107370,2708.937487,3.039548
1,AGC_001,2022,2,2022-02-01,14.697198,21.410572,6.613323,2708.937487,3.039548
2,AGC_001,2022,3,2022-03-01,4.974521,24.541271,8.275019,2708.937487,3.039548
3,AGC_001,2022,4,2022-04-01,47.755169,25.377922,11.046246,2708.937487,3.039548
4,AGC_001,2022,5,2022-05-01,45.257622,26.147617,11.043602,2708.937487,3.039548



Se creó df_ambiente_mensual en memoria.
No se modificaron las tablas de origen ni se guardaron archivos.


In [33]:
# Bloque 30: Resumir índices por parcela y mes, conservando separados los sensores.

satelite_mensual_por_sensor = {}
resumen_satelite_mensual = []

fuentes_satelitales = {
    "basico": df_basico_preparado,
    "pro": df_pro_preparado,
}

for fuente, tabla in fuentes_satelitales.items():
    for sensor, grupo in tabla.groupby("sensor"):
        grupo = grupo.copy()

        grupo["fecha_mes"] = (
            grupo["fecha_captura"]
            .dt.to_period("M")
            .dt.to_timestamp()
        )

        columnas_promedio = [
            c for c in grupo.columns if c.endswith("_promedio")
        ]

        if sensor == "Landsat":
            columnas_promedio = [
                c for c in columnas_promedio
                if c.startswith("vi6t_")
            ]
        elif sensor == "Sentinel-2":
            columnas_promedio = [
                c for c in columnas_promedio
                if not c.startswith("vi6t_")
            ]

        assert columnas_promedio, (
            f"No se encontraron índices para {fuente} / {sensor}."
        )

        clave = ["ID_POLIGONO", "fecha_mes"]
        agrupado = grupo.groupby(clave)

        mensual = agrupado[columnas_promedio].agg(["mean", "count"])

        mensual.columns = [
            f"{columna}_{'media_mensual' if estadistico == 'mean' else 'n_obs'}"
            for columna, estadistico in mensual.columns
        ]

        calidad = agrupado.agg(
            capturas_totales=("fecha_captura", "size"),
            capturas_nubosidad_100=(
                "porcentaje_nubosidad",
                lambda serie: int(serie.eq(100).sum())
            ),
            nubosidad_media=("porcentaje_nubosidad", "mean"),
        )

        mensual = mensual.join(
            calidad, validate="one_to_one"
        ).reset_index()

        nombre_tabla = f"{fuente}_{sensor}"
        satelite_mensual_por_sensor[nombre_tabla] = mensual

        columnas_medias = [
            c for c in mensual.columns
            if c.endswith("_media_mensual")
        ]

        resumen_satelite_mensual.append({
            "tabla": nombre_tabla,
            "filas_parcela_mes": len(mensual),
            "parcelas": mensual["ID_POLIGONO"].nunique(),
            "meses_presentes": mensual["fecha_mes"].nunique(),
            "indices": len(columnas_medias),
            "medias_mensuales_NA": int(
                mensual[columnas_medias].isna().sum().sum()
            ),
            "filas_sin_ninguna_media": int(
                mensual[columnas_medias].isna().all(axis=1).sum()
            ),
        })

resumen_mensual_satelital = pd.DataFrame(resumen_satelite_mensual)

display(resumen_mensual_satelital)

print("\nTablas mensuales creadas:")
for nombre in satelite_mensual_por_sensor:
    print(nombre)

print("\nNo se cruzó todavía satélite con clima ni con rendimiento.")
print("Las bases de observaciones originales y preparadas siguen intactas.")


,tabla,filas_parcela_mes,parcelas,meses_presentes,indices,medias_mensuales_NA,filas_sin_ninguna_media
0,basico_Landsat,9456,197,48,1,163,163
1,basico_Sentinel-2,9456,197,48,22,858,39
2,pro_Planet,2364,197,12,4,0,0



Tablas mensuales creadas:
basico_Landsat
basico_Sentinel-2
pro_Planet

No se cruzó todavía satélite con clima ni con rendimiento.
Las bases de observaciones originales y preparadas siguen intactas.


In [34]:
# Bloque 31: Integrar satélite, clima y topografía por parcela y mes sin imputar NAs.

clave_mensual = ["ID_POLIGONO", "fecha_mes"]

df_base_mensual = df_ambiente_mensual.copy()
resumen_cruces_satelitales = []

assert not df_base_mensual.duplicated(clave_mensual).any(), (
    "La tabla ambiental tiene claves repetidas."
)

claves_ambiente = pd.MultiIndex.from_frame(
    df_base_mensual[clave_mensual]
)

for nombre, mensual in satelite_mensual_por_sensor.items():
    assert not mensual.duplicated(clave_mensual).any(), (
        f"Hay claves repetidas en {nombre}."
    )

    claves_satelite = pd.MultiIndex.from_frame(
        mensual[clave_mensual]
    )

    assert claves_satelite.difference(claves_ambiente).empty, (
        f"{nombre} contiene claves fuera de la tabla ambiental."
    )

    prefijo = nombre.lower().replace("-", "_")

    nombres_nuevos = {
        columna: f"{prefijo}__{columna}"
        for columna in mensual.columns
        if columna not in clave_mensual
    }

    mensual_con_prefijo = mensual.rename(columns=nombres_nuevos)

    filas_antes = len(df_base_mensual)

    df_base_mensual = df_base_mensual.merge(
        mensual_con_prefijo,
        on=clave_mensual,
        how="left",
        validate="one_to_one",
        indicator="_estado_cruce"
    )

    columna_presencia = f"{prefijo}__registro_mensual_presente"

    df_base_mensual[columna_presencia] = (
        df_base_mensual["_estado_cruce"].eq("both")
    )

    columnas_medias = [
        nombres_nuevos[columna]
        for columna in mensual.columns
        if columna.endswith("_media_mensual")
    ]

    fuente_presente = df_base_mensual[columna_presencia]

    resumen_cruces_satelitales.append({
        "fuente": nombre,
        "filas_con_registro_mensual": int(fuente_presente.sum()),
        "filas_sin_registro_mensual": int((~fuente_presente).sum()),
        "registros_presentes_sin_medias": int(
            (
                fuente_presente
                & df_base_mensual[columnas_medias].isna().all(axis=1)
            ).sum()
        ),
    })

    df_base_mensual = df_base_mensual.drop(
        columns="_estado_cruce"
    )

    assert len(df_base_mensual) == filas_antes, (
        f"El cruce con {nombre} cambió el número de filas."
    )

assert not df_base_mensual.duplicated(clave_mensual).any()

resumen_base_mensual = pd.DataFrame([{
    "filas": len(df_base_mensual),
    "parcelas": df_base_mensual["ID_POLIGONO"].nunique(),
    "meses": df_base_mensual["fecha_mes"].nunique(),
    "columnas": df_base_mensual.shape[1],
}])

resumen_cruces = pd.DataFrame(resumen_cruces_satelitales)

print("Dimensiones de la base integrada:")
display(resumen_base_mensual)

print("\nResultado de los cruces por fuente:")
display(resumen_cruces)

print("\nNo se imputaron NAs ni se agregó rendimiento.")
print("La base integrada está en df_base_mensual, solo en memoria.")


Dimensiones de la base integrada:


,filas,parcelas,meses,columnas
0,9456,197,48,75



Resultado de los cruces por fuente:


,fuente,filas_con_registro_mensual,filas_sin_registro_mensual,registros_presentes_sin_medias
0,basico_Landsat,9456,0,163
1,basico_Sentinel-2,9456,0,39
2,pro_Planet,2364,7092,0



No se imputaron NAs ni se agregó rendimiento.
La base integrada está en df_base_mensual, solo en memoria.


In [35]:
# Bloque 32: Separar abril-octubre de 2025 y revisar disponibilidad de índices por sensor.

inicio_ciclo = pd.Timestamp("2025-04-01")
fin_ciclo_exclusivo = pd.Timestamp("2025-11-01")

df_ciclo_mensual = df_base_mensual.loc[
    df_base_mensual["fecha_mes"].ge(inicio_ciclo)
    & df_base_mensual["fecha_mes"].lt(fin_ciclo_exclusivo)
].copy()

assert not df_ciclo_mensual.duplicated(
    ["ID_POLIGONO", "fecha_mes"]
).any(), "Hay claves repetidas en el ciclo."

meses_por_parcela = (
    df_ciclo_mensual.groupby("ID_POLIGONO")["fecha_mes"].nunique()
)

assert set(df_ciclo_mensual["ID_POLIGONO"]) == set(
    df_rendimiento["ID_POLIGONO"]
), "No están representadas las mismas parcelas."

assert meses_por_parcela.eq(7).all(), (
    "Alguna parcela no tiene los siete meses del ciclo."
)

resumen_ciclo = pd.DataFrame([{
    "filas": len(df_ciclo_mensual),
    "parcelas": df_ciclo_mensual["ID_POLIGONO"].nunique(),
    "meses": df_ciclo_mensual["fecha_mes"].nunique(),
    "fecha_inicial": df_ciclo_mensual["fecha_mes"].min(),
    "fecha_final": df_ciclo_mensual["fecha_mes"].max(),
}])

disponibilidad_ciclo = []

for prefijo in [
    "basico_landsat",
    "basico_sentinel_2",
    "pro_planet",
]:
    columnas_medias = [
        c for c in df_ciclo_mensual.columns
        if c.startswith(f"{prefijo}__")
        and c.endswith("_media_mensual")
    ]

    presente = df_ciclo_mensual[
        f"{prefijo}__registro_mensual_presente"
    ]

    medias = df_ciclo_mensual[columnas_medias]

    disponibilidad_ciclo.append({
        "fuente": prefijo,
        "indices": len(columnas_medias),
        "filas_sin_registro": int((~presente).sum()),
        "filas_sin_ninguna_media": int(
            medias.isna().all(axis=1).sum()
        ),
        "celdas_NA_en_medias": int(medias.isna().sum().sum()),
    })

resumen_disponibilidad_ciclo = pd.DataFrame(disponibilidad_ciclo)

print("Base mensual del ciclo:")
display(resumen_ciclo)

print("\nDisponibilidad satelital dentro del ciclo:")
display(resumen_disponibilidad_ciclo)

print("\nSe creó df_ciclo_mensual sin modificar df_base_mensual.")
print("No se imputaron valores ni se agregó rendimiento.")


Base mensual del ciclo:


,filas,parcelas,meses,fecha_inicial,fecha_final
0,1379,197,7,2025-04-01,2025-10-01



Disponibilidad satelital dentro del ciclo:


,fuente,indices,filas_sin_registro,filas_sin_ninguna_media,celdas_NA_en_medias
0,basico_landsat,1,0,8,8
1,basico_sentinel_2,22,0,0,0
2,pro_planet,4,0,0,0



Se creó df_ciclo_mensual sin modificar df_base_mensual.
No se imputaron valores ni se agregó rendimiento.


In [36]:
# Bloque 33: Crear una fila por parcela con resúmenes del ciclo y rendimiento.

agrupado_ciclo = df_ciclo_mensual.groupby("ID_POLIGONO")

assert (
    agrupado_ciclo[["elevacion_m", "pendiente_grados"]]
    .nunique(dropna=False)
    .eq(1)
    .all()
    .all()
), "La topografía cambia entre meses de una misma parcela."

resumen_ambiental_ciclo = agrupado_ciclo.agg(
    precipitacion_acumulada_mm=(
        "precipitacion_mm",
        lambda serie: serie.sum(min_count=7)
    ),
    tmax_media_ciclo_c=("tmax_c", "mean"),
    tmin_media_ciclo_c=("tmin_c", "mean"),
    elevacion_m=("elevacion_m", "first"),
    pendiente_grados=("pendiente_grados", "first"),
)

columnas_indices_mensuales = [
    c for c in df_ciclo_mensual.columns
    if c.endswith("_media_mensual")
]

medias_indices_ciclo = (
    agrupado_ciclo[columnas_indices_mensuales].mean()
)

meses_indices_ciclo = (
    agrupado_ciclo[columnas_indices_mensuales].count()
)

nombres_medias_ciclo = {
    c: c.replace("_promedio_media_mensual", "__media_ciclo")
    for c in columnas_indices_mensuales
}

nombres_meses_ciclo = {
    c: c.replace("_promedio_media_mensual", "__meses_disponibles")
    for c in columnas_indices_mensuales
}

medias_indices_ciclo = medias_indices_ciclo.rename(
    columns=nombres_medias_ciclo
)

meses_indices_ciclo = meses_indices_ciclo.rename(
    columns=nombres_meses_ciclo
)

resumen_por_parcela = (
    resumen_ambiental_ciclo
    .join(medias_indices_ciclo, validate="one_to_one")
    .join(meses_indices_ciclo, validate="one_to_one")
    .reset_index()
)

assert set(resumen_por_parcela["ID_POLIGONO"]) == set(
    df_rendimiento["ID_POLIGONO"]
), "Las parcelas del resumen no coinciden con rendimiento."

df_eda_parcela = df_rendimiento.merge(
    resumen_por_parcela,
    on="ID_POLIGONO",
    how="left",
    validate="one_to_one"
)

columnas_predictoras_eda = [
    "AREA_HA",
    "precipitacion_acumulada_mm",
    "tmax_media_ciclo_c",
    "tmin_media_ciclo_c",
    "elevacion_m",
    "pendiente_grados",
] + list(nombres_medias_ciclo.values())

resumen_eda_parcela = pd.DataFrame([{
    "filas": len(df_eda_parcela),
    "IDs_unicos": df_eda_parcela["ID_POLIGONO"].nunique(),
    "columnas": df_eda_parcela.shape[1],
    "predictoras_para_explorar": len(columnas_predictoras_eda),
    "parcelas_con_rendimiento": int(
        df_eda_parcela["RENDIMIENTO_T_HA"].notna().sum()
    ),
    "parcelas_sin_rendimiento": int(
        df_eda_parcela["RENDIMIENTO_T_HA"].isna().sum()
    ),
    "NA_en_predictoras": int(
        df_eda_parcela[columnas_predictoras_eda].isna().sum().sum()
    ),
}])

columnas_cobertura = list(nombres_meses_ciclo.values())

cobertura_incompleta = df_eda_parcela.loc[
    df_eda_parcela[columnas_cobertura].lt(7).any(axis=1),
    ["ID_POLIGONO"] + columnas_cobertura
]

print("Tabla exploratoria por parcela:")
display(resumen_eda_parcela)

print("\nParcelas con menos de siete meses en algún índice:")
display(cobertura_incompleta)

print("\nNo se imputaron nuevos valores ni se calcularon correlaciones.")
print("El rendimiento de las parcelas de predicción sigue vacío.")

Tabla exploratoria por parcela:


,filas,IDs_unicos,columnas,predictoras_para_explorar,parcelas_con_rendimiento,parcelas_sin_rendimiento,NA_en_predictoras
0,197,197,63,33,138,59,0



Parcelas con menos de siete meses en algún índice:


,ID_POLIGONO,basico_landsat__vi6t__meses_disponibles,basico_sentinel_2__ndvi__meses_disponibles,basico_sentinel_2__evi__meses_disponibles,basico_sentinel_2__crc__meses_disponibles,basico_sentinel_2__mcrc__meses_disponibles,basico_sentinel_2__ndsvi__meses_disponibles,basico_sentinel_2__srndi__meses_disponibles,basico_sentinel_2__mra__meses_disponibles,basico_sentinel_2__lai__meses_disponibles,...,basico_sentinel_2__dswi2__meses_disponibles,basico_sentinel_2__dswi3__meses_disponibles,basico_sentinel_2__dswi4__meses_disponibles,basico_sentinel_2__dswi5__meses_disponibles,basico_sentinel_2__nddi__meses_disponibles,basico_sentinel_2__enddi__meses_disponibles,pro_planet__ndvi__meses_disponibles,pro_planet__evi__meses_disponibles,pro_planet__lai__meses_disponibles,pro_planet__msavi__meses_disponibles
0,AGC_001,6,7,7,7,7,7,7,7,7,...,7,7,7,7,7,7,7,7,7,7
6,AGC_007,6,7,7,7,7,7,7,7,7,...,7,7,7,7,7,7,7,7,7,7
33,AGC_034,6,7,7,7,7,7,7,7,7,...,7,7,7,7,7,7,7,7,7,7
66,AGC_067,6,7,7,7,7,7,7,7,7,...,7,7,7,7,7,7,7,7,7,7
77,AGC_078,6,7,7,7,7,7,7,7,7,...,7,7,7,7,7,7,7,7,7,7
79,AGC_080,6,7,7,7,7,7,7,7,7,...,7,7,7,7,7,7,7,7,7,7
117,AGC_118,6,7,7,7,7,7,7,7,7,...,7,7,7,7,7,7,7,7,7,7
139,AGC_140,6,7,7,7,7,7,7,7,7,...,7,7,7,7,7,7,7,7,7,7



No se imputaron nuevos valores ni se calcularon correlaciones.
El rendimiento de las parcelas de predicción sigue vacío.


In [37]:
# Bloque 33.1: Exportar la tabla integrada por parcela y mostrar una vista previa.

ruta_csv_tabla_parcela = (
    directorio_actual / "tabla_integrada_parcela_ciclo_2025.csv"
)

df_eda_parcela.to_csv(
    ruta_csv_tabla_parcela,
    index=False,
    encoding="utf-8-sig"
)

print(f"CSV guardado en: {ruta_csv_tabla_parcela.resolve()}")
print(
    f"Filas: {df_eda_parcela.shape[0]} | "
    f"Columnas: {df_eda_parcela.shape[1]}"
)

print("\nVista previa: primeras 10 parcelas y todas las columnas")

with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", 45
):
    display(df_eda_parcela.head(10))

CSV guardado en: C:\Users\navae\OneDrive\Escritorio\Tec\Semestre 7\AgroCebada\AgroCebada2026\notebooks\01_eda_erick\tabla_integrada_parcela_ciclo_2025.csv
Filas: 197 | Columnas: 63

Vista previa: primeras 10 parcelas y todas las columnas


,ID_POLIGONO,AREA_HA,RENDIMIENTO_T_HA,CONJUNTO,precipitacion_acumulada_mm,tmax_media_ciclo_c,tmin_media_ciclo_c,elevacion_m,pendiente_grados,basico_landsat__vi6t__media_ciclo,basico_sentinel_2__ndvi__media_ciclo,basico_sentinel_2__evi__media_ciclo,basico_sentinel_2__crc__media_ciclo,basico_sentinel_2__mcrc__media_ciclo,basico_sentinel_2__ndsvi__media_ciclo,basico_sentinel_2__srndi__media_ciclo,basico_sentinel_2__mra__media_ciclo,basico_sentinel_2__lai__media_ciclo,basico_sentinel_2__fapar__media_ciclo,basico_sentinel_2__savi__media_ciclo,basico_sentinel_2__ndti__media_ciclo,basico_sentinel_2__sti__media_ciclo,basico_sentinel_2__ndwi__media_ciclo,basico_sentinel_2__endwi__media_ciclo,basico_sentinel_2__msi__media_ciclo,basico_sentinel_2__emsi__media_ciclo,basico_sentinel_2__dswi2__media_ciclo,basico_sentinel_2__dswi3__media_ciclo,basico_sentinel_2__dswi4__media_ciclo,basico_sentinel_2__dswi5__media_ciclo,basico_sentinel_2__nddi__media_ciclo,basico_sentinel_2__enddi__media_ciclo,pro_planet__ndvi__media_ciclo,pro_planet__evi__media_ciclo,pro_planet__lai__media_ciclo,pro_planet__msavi__media_ciclo,basico_landsat__vi6t__meses_disponibles,basico_sentinel_2__ndvi__meses_disponibles,basico_sentinel_2__evi__meses_disponibles,basico_sentinel_2__crc__meses_disponibles,basico_sentinel_2__mcrc__meses_disponibles,basico_sentinel_2__ndsvi__meses_disponibles,basico_sentinel_2__srndi__meses_disponibles,basico_sentinel_2__mra__meses_disponibles,basico_sentinel_2__lai__meses_disponibles,basico_sentinel_2__fapar__meses_disponibles,basico_sentinel_2__savi__meses_disponibles,basico_sentinel_2__ndti__meses_disponibles,basico_sentinel_2__sti__meses_disponibles,basico_sentinel_2__ndwi__meses_disponibles,basico_sentinel_2__endwi__meses_disponibles,basico_sentinel_2__msi__meses_disponibles,basico_sentinel_2__emsi__meses_disponibles,basico_sentinel_2__dswi2__meses_disponibles,basico_sentinel_2__dswi3__meses_disponibles,basico_sentinel_2__dswi4__meses_disponibles,basico_sentinel_2__dswi5__meses_disponibles,basico_sentinel_2__nddi__meses_disponibles,basico_sentinel_2__enddi__meses_disponibles,pro_planet__ndvi__meses_disponibles,pro_planet__evi__meses_disponibles,pro_planet__lai__meses_disponibles,pro_planet__msavi__meses_disponibles
0,AGC_001,7.61,3.50,ENTRENAMIENTO,1360.919123,23.349517,11.596077,2708.937487,3.039548,-0.622417,0.313874,0.264809,0.278330,0.261352,0.258630,0.103577,0.024913,0.261049,0.299672,0.197920,0.159838,1.425160,0.078395,0.215533,0.917356,0.726415,1.761469,1.756857,1.027202,1.210719,0.044560,0.100220,0.426584,0.281466,1.493696,0.277414,6,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7
1,AGC_002,2.31,4.63,ENTRENAMIENTO,961.350809,23.780726,13.328253,2611.461295,0.597507,-0.619117,0.401613,0.343580,0.342955,0.325674,0.352182,0.165481,0.026280,0.416130,0.369480,0.255860,0.198869,1.550987,0.081230,0.258602,0.915685,0.680946,2.028185,2.179108,1.100446,1.224713,0.061049,0.131473,0.494198,0.331188,1.696134,0.322235,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7
2,AGC_003,18.68,NaN,PREDICCION,1335.090225,23.801091,13.277613,2642.600950,2.278437,-0.640738,0.389077,0.292542,0.323601,0.330351,0.350131,0.149196,0.027095,0.342137,0.327565,0.224363,0.212458,1.594649,0.068250,0.252690,0.960929,0.694036,2.113214,2.165393,1.087452,1.233071,0.060173,0.135827,0.481336,0.290618,1.678324,0.289328,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7
3,AGC_004,8.00,3.00,ENTRENAMIENTO,887.946089,23.791028,12.122947,2564.199598,1.345183,-0.625405,0.313369,0.259849,0.324109,0.320695,0.316936,0.169082,0.024921,0.240726,0.275418,0.195336,0.156844,1.400704,0.010962,0.155323,1.042909,0.810468,2.016471,1.999351,1.013638,1.078318,0.069760,0.132659,0.421825,0.271053,1.496225,0.261250,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7
4,AGC_005,2.27,5.16,ENTRENAMIENTO,793.730323,23.513994,13.248211,2623.981209,1.432019,-0.636631,0.404308,0.355495,0.373332,0.368249,0.413888,0.246010,0.027069,0.522468,0.348149,0.259399,0.185570,1.514206,0.026461,0.181324,

In [38]:
# Bloque 33.2: Pasar el ciclo mensual a columnas por mes, sin imputar faltantes.

clave_mensual = ["ID_POLIGONO", "fecha_mes"]

assert not df_ciclo_mensual.duplicated(clave_mensual).any(), (
    "Hay más de una fila por parcela y mes."
)

fechas_esperadas = pd.date_range(
    "2025-04-01",
    "2025-10-01",
    freq="MS"
)

assert set(df_ciclo_mensual["fecha_mes"]) == set(fechas_esperadas), (
    "Las fechas no coinciden con abril-octubre de 2025."
)

assert (
    df_ciclo_mensual.groupby("ID_POLIGONO")["fecha_mes"]
    .nunique()
    .eq(7)
    .all()
), "Alguna parcela no tiene los siete registros mensuales."

columnas_clima_mensual = [
    "precipitacion_mm",
    "tmax_c",
    "tmin_c",
]

columnas_indices_mensuales = [
    c for c in df_ciclo_mensual.columns
    if c.endswith("_media_mensual")
]

columnas_conteos_mensuales = [
    c for c in df_ciclo_mensual.columns
    if c.endswith("_n_obs")
]

columnas_presencia_mensual = [
    c for c in df_ciclo_mensual.columns
    if c.endswith("__registro_mensual_presente")
]

columnas_a_expandir = (
    columnas_clima_mensual
    + columnas_indices_mensuales
    + columnas_conteos_mensuales
    + columnas_presencia_mensual
)

df_para_expandir = df_ciclo_mensual[
    ["ID_POLIGONO", "fecha_mes"] + columnas_a_expandir
].copy()

tabla_ancha_mensual = df_para_expandir.pivot(
    index="ID_POLIGONO",
    columns="fecha_mes",
    values=columnas_a_expandir
)

orden_columnas = pd.MultiIndex.from_product(
    [columnas_a_expandir, fechas_esperadas],
    names=tabla_ancha_mensual.columns.names
)

tabla_ancha_mensual = tabla_ancha_mensual.reindex(
    columns=orden_columnas
)

nombres_columnas = []

for variable, fecha in tabla_ancha_mensual.columns:
    nombre_base = (
        variable
        .replace("_promedio_media_mensual", "__media_mensual")
        .replace("_promedio_n_obs", "__n_obs")
    )

    nombres_columnas.append(
        f"{nombre_base}__{fecha.strftime('%Y_%m')}"
    )

tabla_ancha_mensual.columns = nombres_columnas

df_parcela_meses_base = tabla_ancha_mensual.reset_index()

assert df_parcela_meses_base["ID_POLIGONO"].is_unique
assert df_parcela_meses_base.columns.is_unique

assert set(df_parcela_meses_base["ID_POLIGONO"]) == set(
    df_ciclo_mensual["ID_POLIGONO"]
), "Se perdieron parcelas durante la transformación."

columnas_medias_expandidas = [
    c for c in df_parcela_meses_base.columns
    if "__media_mensual__" in c
]

na_indices_antes = int(
    df_ciclo_mensual[columnas_indices_mensuales].isna().sum().sum()
)

na_indices_despues = int(
    df_parcela_meses_base[columnas_medias_expandidas]
    .isna()
    .sum()
    .sum()
)

assert na_indices_antes == na_indices_despues, (
    "Cambió el número de faltantes de los índices."
)

resumen_expansion_mensual = pd.DataFrame([{
    "filas": len(df_parcela_meses_base),
    "parcelas": df_parcela_meses_base["ID_POLIGONO"].nunique(),
    "meses_expandidos": len(fechas_esperadas),
    "variables_climaticas": len(columnas_clima_mensual),
    "indices_satelitales": len(columnas_indices_mensuales),
    "columnas_conteos_originales": len(columnas_conteos_mensuales),
    "indicadores_presencia_originales": len(columnas_presencia_mensual),
    "columnas_totales": df_parcela_meses_base.shape[1],
    "NA_indices_antes": na_indices_antes,
    "NA_indices_despues": na_indices_despues,
}])

print("Resumen de la transformación:")
display(resumen_expansion_mensual)

print("\nEjemplo: NDVI de Sentinel-2 separado por mes")

columnas_ejemplo = ["ID_POLIGONO"] + [
    c for c in df_parcela_meses_base.columns
    if c.startswith("basico_sentinel_2__ndvi__media_mensual__")
]

display(df_parcela_meses_base[columnas_ejemplo].head())

print("\nLa nueva tabla está en df_parcela_meses_base.")
print("No se modificaron las tablas anteriores ni se imputaron valores.")


Resumen de la transformación:


,filas,parcelas,meses_expandidos,variables_climaticas,indices_satelitales,columnas_conteos_originales,indicadores_presencia_originales,columnas_totales,NA_indices_antes,NA_indices_despues
0,197,197,7,3,27,27,3,421,8,8



Ejemplo: NDVI de Sentinel-2 separado por mes


,ID_POLIGONO,basico_sentinel_2__ndvi__media_mensual__2025_04,basico_sentinel_2__ndvi__media_mensual__2025_05,basico_sentinel_2__ndvi__media_mensual__2025_06,basico_sentinel_2__ndvi__media_mensual__2025_07,basico_sentinel_2__ndvi__media_mensual__2025_08,basico_sentinel_2__ndvi__media_mensual__2025_09,basico_sentinel_2__ndvi__media_mensual__2025_10
0,AGC_001,0.12325,0.10775,0.1615,0.245833,0.420714,0.5935,0.544571
1,AGC_002,0.143875,0.1475,0.365,0.669667,0.6245,0.459,0.40175
2,AGC_003,0.233875,0.156,0.176,0.408333,0.643333,0.589,0.517
3,AGC_004,0.140125,0.125833,0.172,0.3342,0.455857,0.499167,0.4664
4,AGC_005,0.178625,0.1765,0.232333,0.3964,0.4915,0.6838,0.671



La nueva tabla está en df_parcela_meses_base.
No se modificaron las tablas anteriores ni se imputaron valores.


In [39]:
# Bloque 33.2.1: Identificar huecos mensuales de VI6T y su contexto.

columna_vi6t = "basico_landsat__vi6t_promedio_media_mensual"

columnas_contexto_huecos = [
    "ID_POLIGONO",
    "fecha_mes",
    columna_vi6t,
    "basico_landsat__vi6t_promedio_n_obs",
    "basico_landsat__capturas_totales",
    "basico_landsat__capturas_nubosidad_100",
    "basico_landsat__nubosidad_media",
    "basico_landsat__registro_mensual_presente",
]

df_contexto_vi6t = df_ciclo_mensual.sort_values(
    ["ID_POLIGONO", "fecha_mes"]
).copy()

por_parcela = df_contexto_vi6t.groupby("ID_POLIGONO")[
    columna_vi6t
]

# Son los meses inmediatamente anterior y posterior dentro del ciclo.
df_contexto_vi6t["vi6t_mes_anterior"] = por_parcela.shift(1)
df_contexto_vi6t["vi6t_mes_siguiente"] = por_parcela.shift(-1)

huecos_vi6t = df_contexto_vi6t.loc[
    df_contexto_vi6t[columna_vi6t].isna(),
    columnas_contexto_huecos
    + ["vi6t_mes_anterior", "vi6t_mes_siguiente"]
].copy()

huecos_vi6t["todas_capturas_nubosidad_100"] = (
    huecos_vi6t["basico_landsat__capturas_totales"].gt(0)
    & huecos_vi6t["basico_landsat__capturas_nubosidad_100"].eq(
        huecos_vi6t["basico_landsat__capturas_totales"]
    )
)

huecos_vi6t["ambos_meses_vecinos_disponibles"] = (
    huecos_vi6t[
        ["vi6t_mes_anterior", "vi6t_mes_siguiente"]
    ].notna().all(axis=1)
)

print("Huecos mensuales de VI6T y contexto:")
with pd.option_context("display.max_columns", None):
    display(huecos_vi6t)

print("\nHuecos por mes:")
display(
    huecos_vi6t.groupby("fecha_mes")
    .size()
    .rename("huecos")
    .reset_index()
)

print("\nNo se imputó ningún valor.")
print(
    "Un vecino vacío en abril u octubre puede deberse "
    "al límite del ciclo, no a un faltante en la base histórica."
)


Huecos mensuales de VI6T y contexto:


,ID_POLIGONO,fecha_mes,basico_landsat__vi6t_promedio_media_mensual,basico_landsat__vi6t_promedio_n_obs,basico_landsat__capturas_totales,basico_landsat__capturas_nubosidad_100,basico_landsat__nubosidad_media,basico_landsat__registro_mensual_presente,vi6t_mes_anterior,vi6t_mes_siguiente,todas_capturas_nubosidad_100,ambos_meses_vecinos_disponibles
44,AGC_001,2025-09-01,NaN,0,4,4,100.0,True,-0.61825,-0.644000,True,True
332,AGC_007,2025-09-01,NaN,0,4,4,100.0,True,-0.64900,-0.612667,True,True
1626,AGC_034,2025-07-01,NaN,0,3,3,100.0,True,-0.62900,-0.641667,True,True
3212,AGC_067,2025-09-01,NaN,0,4,4,100.0,True,-0.64450,-0.631000,True,True
3740,AGC_078,2025-09-01,NaN,0,4,4,100.0,True,-0.64200,-0.639000,True,True
3834,AGC_080,2025-07-01,NaN,0,3,3,100.0,True,-0.63750,-0.646333,True,True
5658,AGC_118,2025-07-01,NaN,0,3,3,100.0,True,-0.61400,-0.634667,True,True
6716,AGC_140,2025-09-01,NaN,0,4,4,100.0,True,-0.62850,-0.652000,True,True



Huecos por mes:


,fecha_mes,huecos
0,2025-07-01,3
1,2025-09-01,5



No se imputó ningún valor.
Un vecino vacío en abril u octubre puede deberse al límite del ciclo, no a un faltante en la base histórica.


In [40]:
# Bloque 33.2.2: Evaluar huecos artificiales de VI6T en julio y septiembre.

columna_vi6t = "basico_landsat__vi6t_promedio_media_mensual"

df_prueba_temporal = (
    df_ciclo_mensual[
        ["ID_POLIGONO", "fecha_mes", columna_vi6t]
    ]
    .sort_values(["ID_POLIGONO", "fecha_mes"])
    .copy()
)

assert not df_prueba_temporal.duplicated(
    ["ID_POLIGONO", "fecha_mes"]
).any(), "Hay claves repetidas."

grupo = df_prueba_temporal.groupby("ID_POLIGONO")

df_prueba_temporal["vi6t_anterior"] = (
    grupo[columna_vi6t].shift(1)
)
df_prueba_temporal["vi6t_siguiente"] = (
    grupo[columna_vi6t].shift(-1)
)
df_prueba_temporal["fecha_anterior"] = (
    grupo["fecha_mes"].shift(1)
)
df_prueba_temporal["fecha_siguiente"] = (
    grupo["fecha_mes"].shift(-1)
)

meses_con_huecos_reales = (
    df_prueba_temporal.loc[
        df_prueba_temporal[columna_vi6t].isna(),
        "fecha_mes"
    ].unique()
)

evaluacion_huecos_vi6t = df_prueba_temporal.loc[
    df_prueba_temporal["fecha_mes"].isin(meses_con_huecos_reales)
    & df_prueba_temporal[
        [columna_vi6t, "vi6t_anterior", "vi6t_siguiente"]
    ].notna().all(axis=1)
].copy()

assert not evaluacion_huecos_vi6t.empty, (
    "No hay valores conocidos con ambos vecinos para evaluar."
)

# Interpolación usando la distancia temporal entre fechas mensuales.
fraccion_temporal = (
    (
        evaluacion_huecos_vi6t["fecha_mes"]
        - evaluacion_huecos_vi6t["fecha_anterior"]
    ).dt.total_seconds()
    /
    (
        evaluacion_huecos_vi6t["fecha_siguiente"]
        - evaluacion_huecos_vi6t["fecha_anterior"]
    ).dt.total_seconds()
)

evaluacion_huecos_vi6t["pred_mes_anterior"] = (
    evaluacion_huecos_vi6t["vi6t_anterior"]
)

evaluacion_huecos_vi6t["pred_interpolacion"] = (
    evaluacion_huecos_vi6t["vi6t_anterior"]
    + fraccion_temporal * (
        evaluacion_huecos_vi6t["vi6t_siguiente"]
        - evaluacion_huecos_vi6t["vi6t_anterior"]
    )
)

filas_metricas = []

for fecha, casos in evaluacion_huecos_vi6t.groupby("fecha_mes"):
    for metodo in ["pred_mes_anterior", "pred_interpolacion"]:
        errores = casos[metodo] - casos[columna_vi6t]

        filas_metricas.append({
            "mes": fecha.strftime("%Y-%m"),
            "metodo": metodo,
            "n_casos": len(casos),
            "MAE": errores.abs().mean(),
            "RMSE": (errores.pow(2).mean()) ** 0.5,
            "sesgo_pred_menos_real": errores.mean(),
        })

metricas_huecos_vi6t = pd.DataFrame(filas_metricas)

print("Evaluación de huecos artificiales por mes:")
display(metricas_huecos_vi6t)

print("\nEjemplos de reconstrucción:")
display(
    evaluacion_huecos_vi6t[[
        "ID_POLIGONO",
        "fecha_mes",
        columna_vi6t,
        "pred_mes_anterior",
        "pred_interpolacion",
    ]].head(10)
)

print(
    "\nLos errores están expresados en unidades de VI6T."
    "\nNo se usó rendimiento ni se imputaron los huecos reales."
    "\nCada caso se evaluó como un hueco aislado."
)

Evaluación de huecos artificiales por mes:


,mes,metodo,n_casos,MAE,RMSE,sesgo_pred_menos_real
0,2025-07,pred_mes_anterior,194,0.015302,0.020165,0.010315
1,2025-07,pred_interpolacion,194,0.011075,0.014043,-0.001044
2,2025-09,pred_mes_anterior,192,0.011654,0.015504,-0.000616
3,2025-09,pred_interpolacion,192,0.010686,0.013784,-0.000616



Ejemplos de reconstrucción:


,ID_POLIGONO,fecha_mes,basico_landsat__vi6t_promedio_media_mensual,pred_mes_anterior,pred_interpolacion
42,AGC_001,2025-07-01,-0.657500,-0.59600,-0.606943
90,AGC_002,2025-07-01,-0.610667,-0.58800,-0.620582
92,AGC_002,2025-09-01,-0.626000,-0.65425,-0.642180
138,AGC_003,2025-07-01,-0.661000,-0.61200,-0.632164
140,AGC_003,2025-09-01,-0.645000,-0.65300,-0.648257
186,AGC_004,2025-07-01,-0.625000,-0.61600,-0.621164
188,AGC_004,2025-09-01,-0.623000,-0.62650,-0.633361
234,AGC_005,2025-07-01,-0.639333,-0.61700,-0.633844
236,AGC_005,2025-09-01,-0.640000,-0.65125,-0.645787
282,AGC_006,2025-07-01,-0.617000,-0.61650,-0.619369



Los errores están expresados en unidades de VI6T.
No se usó rendimiento ni se imputaron los huecos reales.
Cada caso se evaluó como un hueco aislado.


In [41]:
# Bloque 33.2.3: Evaluar reconstrucción de VI6T con otros sensores.

import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

columna_vi6t = "basico_landsat__vi6t_promedio_media_mensual"

columnas_otros_sensores = [
    c for c in df_ciclo_mensual.columns
    if c.endswith("_media_mensual")
    and c.startswith(("basico_sentinel_2__", "pro_planet__"))
]

# Mantener exactamente los casos de la evaluación temporal anterior.
df_validacion_sensores = (
    evaluacion_huecos_vi6t[[
        "ID_POLIGONO",
        "fecha_mes",
        columna_vi6t,
        "pred_mes_anterior",
        "pred_interpolacion",
    ]]
    .merge(
        df_ciclo_mensual[
            ["ID_POLIGONO", "fecha_mes"] + columnas_otros_sensores
        ],
        on=["ID_POLIGONO", "fecha_mes"],
        how="left",
        validate="one_to_one"
    )
    .reset_index(drop=True)
)

assert len(df_validacion_sensores) == len(evaluacion_huecos_vi6t)
assert df_validacion_sensores[columna_vi6t].notna().all()

# Identificar el mes sin utilizar VI6T del propio mes.
df_validacion_sensores["es_septiembre"] = (
    df_validacion_sensores["fecha_mes"].dt.month.eq(9).astype(int)
)

columnas_reconstruccion_vi6t = (
    columnas_otros_sensores + ["es_septiembre"]
)

X_reconstruccion = df_validacion_sensores[
    columnas_reconstruccion_vi6t
].astype(float)

y_reconstruccion = df_validacion_sensores[columna_vi6t]

grupos_parcela = df_validacion_sensores["ID_POLIGONO"]

assert not np.isinf(X_reconstruccion.to_numpy()).any()
assert grupos_parcela.nunique() >= 5

modelo_reconstruccion_vi6t = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    Ridge(alpha=10.0)
)

cv_reconstruccion = GroupKFold(n_splits=5)

df_validacion_sensores["pred_otros_sensores_oof"] = (
    cross_val_predict(
        modelo_reconstruccion_vi6t,
        X_reconstruccion,
        y_reconstruccion,
        groups=grupos_parcela,
        cv=cv_reconstruccion
    )
)

filas_comparacion = []

for fecha, casos in df_validacion_sensores.groupby("fecha_mes"):
    for metodo in [
        "pred_mes_anterior",
        "pred_interpolacion",
        "pred_otros_sensores_oof",
    ]:
        errores = casos[metodo] - casos[columna_vi6t]

        filas_comparacion.append({
            "mes": fecha.strftime("%Y-%m"),
            "metodo": metodo,
            "n_casos": len(casos),
            "MAE": errores.abs().mean(),
            "RMSE": errores.pow(2).mean() ** 0.5,
            "sesgo_pred_menos_real": errores.mean(),
        })

comparacion_reconstruccion_vi6t = pd.DataFrame(filas_comparacion)

print("Comparación sobre los mismos valores conocidos:")
display(comparacion_reconstruccion_vi6t)

print(
    "\nOOF: cada estimación se obtuvo sin entrenar con esa parcela."
    "\nLa validación es por parcela, no por bloques espaciales."
    "\nNo se usó rendimiento ni se imputaron los ocho huecos reales."
)


Comparación sobre los mismos valores conocidos:


,mes,metodo,n_casos,MAE,RMSE,sesgo_pred_menos_real
0,2025-07,pred_mes_anterior,194,0.015302,0.020165,0.010315
1,2025-07,pred_interpolacion,194,0.011075,0.014043,-0.001044
2,2025-07,pred_otros_sensores_oof,194,0.008597,0.010984,-0.000118
3,2025-09,pred_mes_anterior,192,0.011654,0.015504,-0.000616
4,2025-09,pred_interpolacion,192,0.010686,0.013784,-0.000616
5,2025-09,pred_otros_sensores_oof,192,0.009970,0.012908,0.000091



OOF: cada estimación se obtuvo sin entrenar con esa parcela.
La validación es por parcela, no por bloques espaciales.
No se usó rendimiento ni se imputaron los ocho huecos reales.


In [42]:
# Bloque 33.3: Integrar metadatos, rendimiento y topografía
# con la tabla de meses separados, conservando los NAs originales.

columna_id = "ID_POLIGONO"

columnas_metadatos = [
    columna_id,
    "AREA_HA",
    "CONJUNTO",
    "RENDIMIENTO_T_HA",
]

columnas_topografia = [
    "elevacion_m",
    "pendiente_grados",
]

# 1. Verificar identificadores antes de cruzar.
for nombre, tabla in {
    "tabla mensual ancha": df_parcela_meses_base,
    "tabla de rendimiento": df_rendimiento,
}.items():
    assert tabla[columna_id].notna().all(), (
        f"Hay identificadores vacíos en {nombre}."
    )
    assert tabla[columna_id].is_unique, (
        f"Hay identificadores repetidos en {nombre}."
    )

ids_mensuales = set(df_parcela_meses_base[columna_id])

assert ids_mensuales == set(df_rendimiento[columna_id]), (
    "Las parcelas de la tabla mensual y rendimiento no coinciden."
)

assert ids_mensuales == set(df_ciclo_mensual[columna_id]), (
    "Las parcelas de la tabla mensual y el ciclo no coinciden."
)

# 2. Verificar que la topografía sea constante entre meses.
variacion_topografia = (
    df_ciclo_mensual
    .groupby(columna_id)[columnas_topografia]
    .nunique(dropna=False)
)

assert variacion_topografia.eq(1).all().all(), (
    "La topografía cambia entre meses para alguna parcela."
)

topografia_por_parcela = (
    df_ciclo_mensual[
        [columna_id] + columnas_topografia
    ]
    .drop_duplicates()
)

assert topografia_por_parcela[columna_id].is_unique, (
    "No se obtuvo una sola fila topográfica por parcela."
)

# Evitar columnas duplicadas o sufijos automáticos en los cruces.
columnas_a_incorporar = (
    columnas_metadatos[1:] + columnas_topografia
)

assert not set(columnas_a_incorporar).intersection(
    df_parcela_meses_base.columns
), "La tabla mensual ya contiene columnas que se van a incorporar."

# 3. Construir la nueva tabla sin sobrescribir las anteriores.
df_parcela_meses = (
    df_rendimiento[columnas_metadatos]
    .merge(
        topografia_por_parcela,
        on=columna_id,
        how="left",
        validate="one_to_one"
    )
    .merge(
        df_parcela_meses_base,
        on=columna_id,
        how="left",
        validate="one_to_one"
    )
)

assert len(df_parcela_meses) == len(df_parcela_meses_base)
assert df_parcela_meses[columna_id].is_unique
assert df_parcela_meses.columns.is_unique

assert df_parcela_meses.shape[1] == (
    df_parcela_meses_base.shape[1] + 5
), "El número de columnas no coincide con lo esperado."

# 4. Comprobar que el rendimiento se conservó exactamente.
pd.testing.assert_series_equal(
    df_rendimiento
    .set_index(columna_id)["RENDIMIENTO_T_HA"]
    .sort_index(),

    df_parcela_meses
    .set_index(columna_id)["RENDIMIENTO_T_HA"]
    .sort_index()
)

# Comprobar que todas las columnas mensuales siguen intactas.
pd.testing.assert_frame_equal(
    df_parcela_meses_base
    .set_index(columna_id)
    .sort_index(),

    df_parcela_meses
    .set_index(columna_id)[
        df_parcela_meses_base.columns.drop(columna_id)
    ]
    .sort_index()
)

# 5. Generar resúmenes para revisión.
columnas_indices_expandidos = [
    c for c in df_parcela_meses.columns
    if "__media_mensual__" in c
]

resumen_integracion_mensual = pd.DataFrame([{
    "filas": len(df_parcela_meses),
    "IDs_unicos": df_parcela_meses[columna_id].nunique(),
    "columnas": df_parcela_meses.shape[1],
    "parcelas_con_rendimiento": int(
        df_parcela_meses["RENDIMIENTO_T_HA"].notna().sum()
    ),
    "parcelas_sin_rendimiento": int(
        df_parcela_meses["RENDIMIENTO_T_HA"].isna().sum()
    ),
    "NA_area": int(
        df_parcela_meses["AREA_HA"].isna().sum()
    ),
    "NA_conjunto": int(
        df_parcela_meses["CONJUNTO"].isna().sum()
    ),
    "NA_topografia": int(
        df_parcela_meses[columnas_topografia].isna().sum().sum()
    ),
    "NA_indices_mensuales": int(
        df_parcela_meses[columnas_indices_expandidos]
        .isna().sum().sum()
    ),
}])

resumen_conjuntos_mensual = (
    df_parcela_meses
    .groupby("CONJUNTO", dropna=False)
    .agg(
        parcelas=(columna_id, "size"),
        con_rendimiento=("RENDIMIENTO_T_HA", "count"),
    )
    .reset_index()
)

resumen_conjuntos_mensual["sin_rendimiento"] = (
    resumen_conjuntos_mensual["parcelas"]
    - resumen_conjuntos_mensual["con_rendimiento"]
)

columnas_vista_previa = (
    columnas_metadatos + columnas_topografia
)

print("Tabla mensual por parcela integrada:")
display(resumen_integracion_mensual)

print("\nDistribución por conjunto:")
display(resumen_conjuntos_mensual)

print("\nPrimeras cinco parcelas: metadatos y topografía")
display(df_parcela_meses[columnas_vista_previa].head())

print("\nNueva tabla: df_parcela_meses.")
print("Los datos mensuales y el rendimiento se conservaron intactos.")
print("No se imputaron valores ni se exportó todavía el CSV.")


Tabla mensual por parcela integrada:


,filas,IDs_unicos,columnas,parcelas_con_rendimiento,parcelas_sin_rendimiento,NA_area,NA_conjunto,NA_topografia,NA_indices_mensuales
0,197,197,426,138,59,0,0,0,8



Distribución por conjunto:


,CONJUNTO,parcelas,con_rendimiento,sin_rendimiento
0,ENTRENAMIENTO,138,138,0
1,PREDICCION,59,0,59



Primeras cinco parcelas: metadatos y topografía


,ID_POLIGONO,AREA_HA,CONJUNTO,RENDIMIENTO_T_HA,elevacion_m,pendiente_grados
0,AGC_001,7.61,ENTRENAMIENTO,3.50,2708.937487,3.039548
1,AGC_002,2.31,ENTRENAMIENTO,4.63,2611.461295,0.597507
2,AGC_003,18.68,PREDICCION,NaN,2642.600950,2.278437
3,AGC_004,8.00,ENTRENAMIENTO,3.00,2564.199598,1.345183
4,AGC_005,2.27,ENTRENAMIENTO,5.16,2623.981209,1.432019



Nueva tabla: df_parcela_meses.
Los datos mensuales y el rendimiento se conservaron intactos.
No se imputaron valores ni se exportó todavía el CSV.


In [43]:
# Bloque 33.4: Verificar tipos, conservar faltantes y exportar la tabla mensual.

import numpy as np

tabla_exportar = df_parcela_meses.copy()

# 1. Clasificar columnas.
columnas_identificacion = ["ID_POLIGONO", "CONJUNTO"]
columnas_objetivo = ["RENDIMIENTO_T_HA"]
columnas_estaticas = [
    "AREA_HA",
    "elevacion_m",
    "pendiente_grados",
]

columnas_clima_ancho = [
    c for c in tabla_exportar.columns
    if c.startswith((
        "precipitacion_mm__",
        "tmax_c__",
        "tmin_c__",
    ))
]

columnas_indices_ancho = [
    c for c in tabla_exportar.columns
    if "__media_mensual__" in c
]

columnas_conteos_ancho = [
    c for c in tabla_exportar.columns
    if "__n_obs__" in c
]

columnas_banderas_ancho = [
    c for c in tabla_exportar.columns
    if "__registro_mensual_presente__" in c
]

grupos_columnas = {
    "identificacion": columnas_identificacion,
    "objetivo": columnas_objetivo,
    "estaticas": columnas_estaticas,
    "clima_mensual": columnas_clima_ancho,
    "indices_mensuales": columnas_indices_ancho,
    "conteos_observaciones": columnas_conteos_ancho,
    "presencia_sensor": columnas_banderas_ancho,
}

columnas_clasificadas = [
    c
    for columnas in grupos_columnas.values()
    for c in columnas
]

assert len(columnas_clasificadas) == len(set(columnas_clasificadas)), (
    "Alguna columna pertenece a más de una categoría."
)

assert set(columnas_clasificadas) == set(tabla_exportar.columns), (
    "Hay columnas sin clasificar."
)

faltantes_antes = tabla_exportar.isna().sum()

# 2. Convertir mediciones a numéricos sin ocultar errores.
columnas_mediciones = (
    columnas_objetivo
    + columnas_estaticas
    + columnas_clima_ancho
    + columnas_indices_ancho
)

for columna in columnas_mediciones:
    tabla_exportar[columna] = pd.to_numeric(
        tabla_exportar[columna],
        errors="raise"
    ).astype("float64")

assert not np.isinf(
    tabla_exportar[columnas_mediciones].to_numpy()
).any(), "Hay valores infinitos en las mediciones."

# 3. Verificar y convertir conteos.
for columna in columnas_conteos_ancho:
    valores = pd.to_numeric(
        tabla_exportar[columna],
        errors="raise"
    )

    presentes = valores.dropna()

    assert np.isfinite(
        presentes.to_numpy(dtype=float)
    ).all(), f"Hay valores no finitos en {columna}."

    assert presentes.ge(0).all(), (
        f"Hay conteos negativos en {columna}."
    )

    assert presentes.eq(presentes.round()).all(), (
        f"Hay conteos no enteros en {columna}."
    )

    tabla_exportar[columna] = valores.astype("Int64")

# 4. Verificar y convertir banderas.
for columna in columnas_banderas_ancho:
    valores = tabla_exportar[columna]

    assert valores.dropna().isin([True, False]).all(), (
        f"Hay valores inesperados en {columna}."
    )

    tabla_exportar[columna] = valores.astype("boolean")

for columna in columnas_identificacion:
    tabla_exportar[columna] = tabla_exportar[columna].astype("string")

# 5. Confirmar que las conversiones no alteraron los faltantes.
pd.testing.assert_series_equal(
    faltantes_antes,
    tabla_exportar.isna().sum()
)

assert tabla_exportar["ID_POLIGONO"].notna().all()
assert tabla_exportar["ID_POLIGONO"].is_unique
assert tabla_exportar.columns.is_unique

assert tabla_exportar["CONJUNTO"].isin(
    ["ENTRENAMIENTO", "PREDICCION"]
).all()

es_entrenamiento = tabla_exportar["CONJUNTO"].eq("ENTRENAMIENTO")
es_prediccion = tabla_exportar["CONJUNTO"].eq("PREDICCION")

assert tabla_exportar.loc[
    es_entrenamiento, "RENDIMIENTO_T_HA"
].notna().all()

assert tabla_exportar.loc[
    es_prediccion, "RENDIMIENTO_T_HA"
].isna().all()

# Aplicar únicamente los tipos verificados a la tabla de esta versión.
df_parcela_meses = tabla_exportar

# Listas separadas: mediciones y documentación de disponibilidad.
columnas_mediciones_candidatas = (
    columnas_estaticas
    + columnas_clima_ancho
    + columnas_indices_ancho
)

columnas_calidad_mensual = (
    columnas_conteos_ancho + columnas_banderas_ancho
)

# 6. Documentar columnas y faltantes.
categoria_por_columna = {
    columna: categoria
    for categoria, columnas in grupos_columnas.items()
    for columna in columnas
}

diccionario_tabla_mensual = pd.DataFrame({
    "columna": df_parcela_meses.columns,
    "categoria": [
        categoria_por_columna[c]
        for c in df_parcela_meses.columns
    ],
    "tipo_pandas": df_parcela_meses.dtypes.astype(str).to_numpy(),
    "faltantes": df_parcela_meses.isna().sum().to_numpy(),
})

resumen_categorias_mensual = (
    diccionario_tabla_mensual
    .groupby("categoria", sort=False)
    .agg(
        columnas=("columna", "size"),
        celdas_faltantes=("faltantes", "sum"),
    )
    .reset_index()
)

# 7. Exportar con nombres diferentes a la versión de medias del ciclo.
ruta_csv_mensual = (
    directorio_actual / "tabla_parcela_meses_2025_sin_imputar.csv"
)

ruta_diccionario_mensual = (
    directorio_actual / "diccionario_tabla_parcela_meses_2025.csv"
)

df_parcela_meses.to_csv(
    ruta_csv_mensual,
    index=False,
    encoding="utf-8-sig",
    na_rep=""
)

diccionario_tabla_mensual.to_csv(
    ruta_diccionario_mensual,
    index=False,
    encoding="utf-8-sig"
)

assert ruta_csv_mensual.is_file()
assert ruta_csv_mensual.stat().st_size > 0
assert ruta_diccionario_mensual.is_file()
assert ruta_diccionario_mensual.stat().st_size > 0

print("Resumen por categoría:")
display(resumen_categorias_mensual)

print("\nColumnas con faltantes:")
display(
    diccionario_tabla_mensual.loc[
        diccionario_tabla_mensual["faltantes"].gt(0)
    ]
)

print(f"\nTabla completa: {ruta_csv_mensual.resolve()}")
print(f"Diccionario: {ruta_diccionario_mensual.resolve()}")

print(
    f"\nTabla: {len(df_parcela_meses)} filas × "
    f"{df_parcela_meses.shape[1]} columnas."
)
print("No se imputaron valores ni se modificó df_eda_parcela.")


Resumen por categoría:


,categoria,columnas,celdas_faltantes
0,identificacion,2,0
1,estaticas,3,0
2,objetivo,1,59
3,clima_mensual,21,0
4,indices_mensuales,189,8
5,conteos_observaciones,189,0
6,presencia_sensor,21,0



Columnas con faltantes:


,columna,categoria,tipo_pandas,faltantes
3,RENDIMIENTO_T_HA,objetivo,float64,59
30,basico_landsat__vi6t__media_mensual__2025_07,indices_mensuales,float64,3
32,basico_landsat__vi6t__media_mensual__2025_09,indices_mensuales,float64,5



Tabla completa: C:\Users\navae\OneDrive\Escritorio\Tec\Semestre 7\AgroCebada\AgroCebada2026\notebooks\01_eda_erick\tabla_parcela_meses_2025_sin_imputar.csv
Diccionario: C:\Users\navae\OneDrive\Escritorio\Tec\Semestre 7\AgroCebada\AgroCebada2026\notebooks\01_eda_erick\diccionario_tabla_parcela_meses_2025.csv

Tabla: 197 filas × 426 columnas.
No se imputaron valores ni se modificó df_eda_parcela.


In [44]:
# Bloque 34: Correlaciones, significancia y ajuste por pruebas múltiples.

import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

objetivo = "RENDIMIENTO_T_HA"
alpha = 0.05
n_permutaciones = 9999
rng = np.random.RandomState(42)

df_eda_etiquetadas = df_eda_parcela.loc[
    df_eda_parcela[objetivo].notna()
].copy()

assert df_eda_etiquetadas["ID_POLIGONO"].is_unique, (
    "Hay parcelas repetidas en la tabla etiquetada."
)

columnas_analisis = columnas_predictoras_eda + [objetivo]

assert all(
    pd.api.types.is_numeric_dtype(df_eda_etiquetadas[c])
    for c in columnas_analisis
), "Alguna variable del análisis no es numérica."

assert not np.isinf(
    df_eda_etiquetadas[columnas_analisis].to_numpy(dtype=float)
).any(), "Hay valores infinitos."

filas_correlaciones = []

for variable in columnas_predictoras_eda:
    pares = df_eda_etiquetadas[
        [variable, objetivo]
    ].dropna()

    x = pares[variable].to_numpy(dtype=float)
    y = pares[objetivo].to_numpy(dtype=float)

    calculable = (
        len(pares) >= 3
        and pares[variable].nunique() > 1
        and pares[objetivo].nunique() > 1
    )

    pearson = p_pearson = spearman = p_spearman = np.nan

    if calculable:
        resultado_pearson = stats.pearsonr(x, y)
        pearson = resultado_pearson.statistic
        p_pearson = resultado_pearson.pvalue

        rangos_x = stats.rankdata(x)
        rangos_y = stats.rankdata(y)

        rangos_x = rangos_x - rangos_x.mean()
        rangos_y = rangos_y - rangos_y.mean()

        denominador = np.sqrt(
            np.sum(rangos_x ** 2) * np.sum(rangos_y ** 2)
        )

        def estadistico_spearman(x_permutado, axis=-1):
            return (
                np.sum(x_permutado * rangos_y, axis=axis)
                / denominador
            )

        resultado_spearman = stats.permutation_test(
            data=(rangos_x,),
            statistic=estadistico_spearman,
            permutation_type="pairings",
            vectorized=True,
            alternative="two-sided",
            n_resamples=n_permutaciones,
            batch=256,
            random_state=rng,
        )

        spearman = resultado_spearman.statistic
        p_spearman = resultado_spearman.pvalue

    filas_correlaciones.append({
        "variable": variable,
        "n_pares": len(pares),
        "valores_distintos": pares[variable].nunique(),
        "pearson": pearson,
        "p_pearson": p_pearson,
        "spearman": spearman,
        "p_spearman_perm": p_spearman,
        "estado": "calculada" if calculable else "no calculable",
    })

correlaciones_rendimiento = pd.DataFrame(filas_correlaciones)

# Ajustar conjuntamente las pruebas de ambos métodos.
p_originales = correlaciones_rendimiento[
    ["p_pearson", "p_spearman_perm"]
].to_numpy(dtype=float)

p_planos = p_originales.ravel()
validos = np.isfinite(p_planos)
p_ajustados = np.full(p_planos.shape, np.nan)

if validos.any():
    p_ajustados[validos] = multipletests(
        p_planos[validos],
        alpha=alpha,
        method="fdr_by"
    )[1]

p_ajustados = p_ajustados.reshape(p_originales.shape)

correlaciones_rendimiento["p_pearson_ajustado_BY"] = (
    p_ajustados[:, 0]
)
correlaciones_rendimiento["p_spearman_ajustado_BY"] = (
    p_ajustados[:, 1]
)

for metodo in ["pearson", "spearman"]:
    columna_p = f"p_{metodo}_ajustado_BY"

    correlaciones_rendimiento[f"significativa_{metodo}_BY"] = (
        correlaciones_rendimiento[columna_p]
        .le(alpha)
        .astype("boolean")
        .mask(correlaciones_rendimiento[columna_p].isna())
    )

correlaciones_rendimiento["abs_spearman"] = (
    correlaciones_rendimiento["spearman"].abs()
)

correlaciones_rendimiento = (
    correlaciones_rendimiento
    .sort_values("abs_spearman", ascending=False, na_position="last")
    .reset_index(drop=True)
)

resumen_significancia = pd.DataFrame([{
    "parcelas_usadas": len(df_eda_etiquetadas),
    "variables_exploradas": len(columnas_predictoras_eda),
    "pruebas_validas_ajustadas": int(validos.sum()),
    "alpha": alpha,
    "permutaciones_spearman": n_permutaciones,
    "significativas_pearson_BY": int(
        correlaciones_rendimiento["significativa_pearson_BY"].sum()
    ),
    "significativas_spearman_BY": int(
        correlaciones_rendimiento["significativa_spearman_BY"].sum()
    ),
}])

display(resumen_significancia)
display(correlaciones_rendimiento)


,parcelas_usadas,variables_exploradas,pruebas_validas_ajustadas,alpha,permutaciones_spearman,significativas_pearson_BY,significativas_spearman_BY
0,138,33,66,0.05,9999,7,4


,variable,n_pares,valores_distintos,pearson,p_pearson,spearman,p_spearman_perm,estado,p_pearson_ajustado_BY,p_spearman_ajustado_BY,significativa_pearson_BY,significativa_spearman_BY,abs_spearman
0,tmin_media_ciclo_c,138,43,0.609574,2.114147e-15,0.542456,0.0002,calculada,6.661934e-13,0.010969,True,True,0.542456
1,precipitacion_acumulada_mm,138,43,-0.468537,6.814315e-09,-0.393708,0.0002,calculada,1.073637e-06,0.010969,True,True,0.393708
2,tmax_media_ciclo_c,138,43,-0.285360,6.924597e-04,-0.363166,0.0002,calculada,2.727531e-02,0.010969,True,True,0.363166
3,basico_sentinel_2__ndvi__media_ciclo,138,138,0.280685,8.542087e-04,0.275997,0.0012,calculada,2.990795e-02,0.037813,True,True,0.275997
4,elevacion_m,138,138,0.289590,5.708749e-04,0.251155,0.0036,calculada,2.569852e-02,0.070900,True,False,0.251155
5,AREA_HA,138,127,-0.078426,3.605454e-01,-0.249140,0.0032,calculada,1.000000e+00,0.070900,False,False,0.249140
6,basico_sentinel_2__ndti__media_ciclo,138,138,0.265221,1.667678e-03,0.248695,0.0034,calculada,4.777325e-02,0.070900,True,False,0.248695
7,basico_sentinel_2__dswi2__media_ciclo,138,138,0.251944,2.873172e-03,0.238791,0.0036,calculada,7.090024e-02,0.070900,False,False,0.238791
8,pendiente_grados,138,138,0.310630,2.088609e-04,0.231998,0.0080,calculada,1.096910e-02,0.126045,True,False,0.231998
9,basico_sentinel_2__savi__media_ciclo,138,138,0.216050,1.092594e-02,0.227724,0.0076,calculada,1.470524e-01,0.126045,False,False,0.227724


In [45]:
# Bloque 35: Revisar valores y perfiles climáticos repetidos en parcelas etiquetadas.

columnas_clima_ciclo = [
    "precipitacion_acumulada_mm",
    "tmax_media_ciclo_c",
    "tmin_media_ciclo_c",
]

resumen_repeticion_clima = []

for variable in columnas_clima_ciclo:
    frecuencias = df_eda_etiquetadas[variable].value_counts(
        dropna=False
    )

    resumen_repeticion_clima.append({
        "variable": variable,
        "parcelas": len(df_eda_etiquetadas),
        "valores_distintos_sin_NA": (
            df_eda_etiquetadas[variable].nunique()
        ),
        "NA": int(df_eda_etiquetadas[variable].isna().sum()),
        "parcelas_en_valores_repetidos": int(
            frecuencias.loc[frecuencias.gt(1)].sum()
        ),
        "max_parcelas_con_mismo_valor": int(frecuencias.max()),
    })

resumen_repeticion_clima = pd.DataFrame(
    resumen_repeticion_clima
)

perfiles_climaticos = (
    df_eda_etiquetadas
    .groupby(columnas_clima_ciclo, dropna=False)
    .agg(
        n_parcelas=("ID_POLIGONO", "size"),
        parcelas=(
            "ID_POLIGONO",
            lambda serie: ", ".join(sorted(serie.astype(str)))
        ),
        rendimiento_medio=("RENDIMIENTO_T_HA", "mean"),
        rendimiento_min=("RENDIMIENTO_T_HA", "min"),
        rendimiento_max=("RENDIMIENTO_T_HA", "max"),
    )
    .reset_index()
    .sort_values("n_parcelas", ascending=False)
    .reset_index(drop=True)
)

resumen_perfiles_climaticos = pd.DataFrame([{
    "parcelas_etiquetadas": len(df_eda_etiquetadas),
    "perfiles_climaticos_distintos": len(perfiles_climaticos),
    "perfiles_compartidos": int(
        perfiles_climaticos["n_parcelas"].gt(1).sum()
    ),
    "parcelas_en_perfiles_compartidos": int(
        perfiles_climaticos.loc[
            perfiles_climaticos["n_parcelas"].gt(1),
            "n_parcelas"
        ].sum()
    ),
    "max_parcelas_por_perfil": int(
        perfiles_climaticos["n_parcelas"].max()
    ),
}])

display(resumen_repeticion_clima)
display(resumen_perfiles_climaticos)
display(perfiles_climaticos.head(10))

print(
    "Los perfiles se compararon por igualdad exacta de valores; "
    "no equivalen necesariamente a celdas o bloques espaciales."
)


,variable,parcelas,valores_distintos_sin_NA,NA,parcelas_en_valores_repetidos,max_parcelas_con_mismo_valor
0,precipitacion_acumulada_mm,138,43,0,107,34
1,tmax_media_ciclo_c,138,43,0,107,34
2,tmin_media_ciclo_c,138,43,0,107,34


,parcelas_etiquetadas,perfiles_climaticos_distintos,perfiles_compartidos,parcelas_en_perfiles_compartidos,max_parcelas_por_perfil
0,138,43,12,107,34


,precipitacion_acumulada_mm,tmax_media_ciclo_c,tmin_media_ciclo_c,n_parcelas,parcelas,rendimiento_medio,rendimiento_min,rendimiento_max
0,961.350809,23.780726,13.328253,34,"AGC_002, AGC_008, AGC_010, AGC_018, AGC_022, A...",4.377941,2.15,5.57
1,793.730323,23.513994,13.248211,29,"AGC_005, AGC_006, AGC_012, AGC_013, AGC_015, A...",4.645172,2.64,5.59
2,887.946089,23.791028,12.122947,11,"AGC_004, AGC_037, AGC_040, AGC_053, AGC_058, A...",3.000000,2.50,3.50
3,1274.556931,23.584246,12.802263,6,"AGC_007, AGC_078, AGC_080, AGC_089, AGC_140, A...",3.200000,3.00,3.50
4,1074.463642,23.734426,12.002433,5,"AGC_019, AGC_046, AGC_103, AGC_114, AGC_144",2.900000,2.50,3.00
5,1335.090225,23.801091,13.277613,5,"AGC_034, AGC_038, AGC_070, AGC_079, AGC_081",3.120000,2.00,3.70
6,1046.503624,23.714047,13.484572,3,"AGC_009, AGC_064, AGC_119",4.566667,4.40,4.79
7,834.477646,23.935325,12.224775,3,"AGC_020, AGC_044, AGC_187",3.333333,3.00,3.50
8,910.267347,23.701606,13.026137,3,"AGC_146, AGC_184, AGC_189",4.360000,4.03,4.58
9,770.927741,23.944563,12.280928,3,"AGC_023, AGC_164, AGC_170",3.000000,3.00,3.00


Los perfiles se compararon por igualdad exacta de valores; no equivalen necesariamente a celdas o bloques espaciales.


In [46]:
# Bloque 36: Comparar correlaciones por parcela y por perfil climático.

objetivo = "RENDIMIENTO_T_HA"

columnas_clima_ciclo = [
    "precipitacion_acumulada_mm",
    "tmax_media_ciclo_c",
    "tmin_media_ciclo_c",
]

assert df_eda_etiquetadas[
    columnas_clima_ciclo + [objetivo]
].notna().all().all(), "Hay faltantes en clima o rendimiento."

df_eda_perfiles = (
    df_eda_etiquetadas
    .groupby(columnas_clima_ciclo, dropna=False)
    .agg(
        n_parcelas=("ID_POLIGONO", "size"),
        rendimiento_medio=(objetivo, "mean"),
    )
    .reset_index()
)

filas_sensibilidad = []

for variable in columnas_clima_ciclo:
    pearson_parcela = df_eda_etiquetadas[variable].corr(
        df_eda_etiquetadas[objetivo],
        method="pearson"
    )

    spearman_parcela = df_eda_etiquetadas[variable].corr(
        df_eda_etiquetadas[objetivo],
        method="spearman"
    )

    pearson_perfil = df_eda_perfiles[variable].corr(
        df_eda_perfiles["rendimiento_medio"],
        method="pearson"
    )

    spearman_perfil = df_eda_perfiles[variable].corr(
        df_eda_perfiles["rendimiento_medio"],
        method="spearman"
    )

    filas_sensibilidad.append({
        "variable": variable,
        "n_parcelas": len(df_eda_etiquetadas),
        "n_perfiles": len(df_eda_perfiles),
        "pearson_por_parcela": pearson_parcela,
        "pearson_por_perfil": pearson_perfil,
        "spearman_por_parcela": spearman_parcela,
        "spearman_por_perfil": spearman_perfil,
    })

sensibilidad_clima = pd.DataFrame(filas_sensibilidad)

print("Comparación descriptiva de asociaciones:")
display(sensibilidad_clima)

print(
    "\nPor parcela: cada parcela aporta una observación."
    "\nPor perfil: cada perfil aporta su rendimiento medio, "
    "sin ponderar por número de parcelas."
    "\nNo se calcularon nuevos valores p."
    "\nNo se modificó df_eda_parcela."
)


Comparación descriptiva de asociaciones:


,variable,n_parcelas,n_perfiles,pearson_por_parcela,pearson_por_perfil,spearman_por_parcela,spearman_por_perfil
0,precipitacion_acumulada_mm,138,43,-0.468537,-0.545848,-0.393708,-0.540259
1,tmax_media_ciclo_c,138,43,-0.285360,-0.244173,-0.363166,-0.294556
2,tmin_media_ciclo_c,138,43,0.609574,0.499882,0.542456,0.514885



Por parcela: cada parcela aporta una observación.
Por perfil: cada perfil aporta su rendimiento medio, sin ponderar por número de parcelas.
No se calcularon nuevos valores p.
No se modificó df_eda_parcela.


In [ ]:
# Bloque 36.1: Exportar tablas procesadas a data/processed/ para el notebook de modelado.
carpeta_procesados = raiz_proyecto / "data" / "processed"
carpeta_procesados.mkdir(parents=True, exist_ok=True)

df_eda_parcela.to_csv(carpeta_procesados / "tabla_integrada_parcela_ciclo_2025.csv", index=False, encoding="utf-8-sig")
df_parcela_meses.to_csv(carpeta_procesados / "tabla_parcela_meses_2025_sin_imputar.csv", index=False, encoding="utf-8-sig", na_rep="")
diccionario_tabla_mensual.to_csv(carpeta_procesados / "diccionario_tabla_parcela_meses_2025.csv", index=False, encoding="utf-8-sig")

df_base_mensual.to_parquet(carpeta_procesados / "df_base_mensual.parquet", index=False)
df_clima_mensual.to_parquet(carpeta_procesados / "df_clima_mensual.parquet", index=False)
df_basico_preparado.to_parquet(carpeta_procesados / "df_basico_preparado.parquet", index=False)
df_pro_preparado.to_parquet(carpeta_procesados / "df_pro_preparado.parquet", index=False)

print("Artefactos del EDA exportados exitosamente en:", carpeta_procesados)
